# DORAnet → enzyme hypotheses → DNA design pipeline

This notebook turns **DORAnet-generated reaction networks** into concrete, review ready **DNA design plans**. Starting from reconstructed biosynthetic pathways, it scores each reaction for feasibility, proposes candidate enzymes, retrieves their protein and DNA sequences and codon optimizes them for a chosen expression host.

## What the pipeline does, step by step
1. **Read target compounds and DORAnet job outputs** — collect every reconstructed pathway from the DORAnet run.
2. **Reconstruct pathway metadata** — parse `*_pathways.txt` / `*_network_pretreated.json` into a tidy per step reaction table and run an optional thermodynamic (ΔH) feasibility check.
3. **Score feasibility with DORA-XGB** — keep only reactions the model rates as feasible.
4. **Map reactions to enzymes** — use Reaction-Center Morgan Fingerprints (RCMFP) to find the closest known natural enzyme reaction and its UniProt ID.
5. **Fetch UniProt metadata** — EC numbers, protein names, organisms, and sequences.
6. **Fetch DNA (CDS) sequences from GenBank** via UniProt cross-references.
7. **Codon-optimize** the selected enzymes for the target host using DNA Chisel.
8. **Export** FASTA + per-pathway figures for downstream design (e.g. Teselagen).


**You only need to edit the single `CONFIG` cell below.** Every file name, directory and run specific setting lives there — the rest of the notebook reads from those variables. Point the paths at your own DORAnet results, adjust the host/optimization settings if needed and run top to bottom.

In [154]:
from pathlib import Path
import glob
import json
import os
import re
import textwrap
import ast
import sys
import time
import hashlib
from io import StringIO, BytesIO

import numpy as np
import pandas as pd
import yaml
import requests

from tqdm.auto import tqdm
tqdm.pandas()

from rdkit import Chem, rdBase
from rdkit.Chem import rdmolfiles, Draw, AllChem

# Pillow -- used by the pathway-plotting cells at the end of the notebook.
from PIL import Image, ImageOps, ImageDraw

from IPython.display import display, Markdown

# DORA-XGB feasibility classifiers (one per cofactor-positioning rule).
from DORA_XGB import DORA_XGB
by_desc_MW_model  = DORA_XGB.feasibility_classifier(cofactor_positioning='by_descending_MW')
by_asc_MW_model   = DORA_XGB.feasibility_classifier(cofactor_positioning='by_ascending_MW')
add_concat_model  = DORA_XGB.feasibility_classifier(cofactor_positioning='add_concat')
add_subtract_model = DORA_XGB.feasibility_classifier(cofactor_positioning='add_subtract')

from Bio.Data import CodonTable

from dnachisel import (
    DnaOptimizationProblem,
    EnforceTranslation,
    EnforceGCContent,
    AvoidPattern,
    MaximizeCAI,
)

## `CONFIG` — the only cell you need to edit

Everything project-specific is data collected here. 

- `dataDir` — root folder holding your DORAnet inputs and reference databases.
- `resultsSubdir` — the specific results run you are processing.
- `doranetSubpath` — path (inside the results run) to the pathway-reconstruction outputs.
- The reference/database file names (rule set, known-enzyme parquet, RCMFP cache).
- `ERGOCHEMICS_SRC` — where your local `ergochemics` source lives.
- `NCBI_EMAIL` — required by NCBI E-utilities; use your own address.

In [155]:
# --- Root locations -------------------------------------------------
dataDir        = '/mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/'
resultsSubdir  = 'combinedEbolaVirus_bestMACAW_allDB_generative/'   
doranetSubpath = 'DORAnet/InoAdeGuoXan/PATHWAY_reconstruction_run2/'      

# --- Input file names (relative to the folders above) ---------------
targetCompoundCsvName = 'combinedEbola_allDORAnetGenerated_Antivirals_wARTprediction_top20.csv'  
doranetRulesetName    = 'DORAnet/JN3604IMT_rules.tsv'               
knownEnzymeParquetName = 'DORAnet/known_enzyme_reactions_union.parquet'  
knownEnzymeRcmfpName   = 'DORAnet/known_rxn_rcmfp_cache.npz'        
doranetMultiGenConfigName = 'config_doranet_multiGen.yaml'          

# --- Local package source ------------------------------------------
ERGOCHEMICS_SRC = '/users/sghosh6/DTRA_project/MACAW/ergochemics/src'

# --- External service settings --------------------------------------
NCBI_EMAIL   = 'sghosh6@lbl.gov'   # REQUIRED by NCBI E-utilities -- use your own email
NCBI_API_KEY = None                # optional; speeds up NCBI requests if set

# --- Codon optimization settings ------------------------------------
targetSpecies = 'e_coli'   # DNA Chisel species code for the expression host
minGc, maxGc  = 0.30, 0.70 # allowed GC-content range
gcWindow      = 50         # window (bp) over which GC content is enforced
stopCodon     = 'TAA'      # stop codon appended to optimized CDS

# --- Derived paths ------------------------------------
dataDir   = os.path.abspath(os.path.expanduser(dataDir))
resultsDir = os.path.join(dataDir, 'Results', resultsSubdir)
DORANETmoleculesDataDir      = os.path.join(resultsDir, doranetSubpath)
DNADesignResultsDir          = os.path.join(resultsDir, 'DNA_design/')
os.makedirs(DNADesignResultsDir, exist_ok=True)

targetCompoundCsvPath = os.path.join(resultsDir, targetCompoundCsvName)
doranetRulesetPath    = os.path.join(dataDir, doranetRulesetName)
knownEnzymeParquetPath = os.path.join(dataDir, knownEnzymeParquetName)
knownEnzymeRcmfpPath   = os.path.join(dataDir, knownEnzymeRcmfpName)

print('Reading DORAnet results from :', DORANETmoleculesDataDir)
print('DNA design results saved at  :', DNADesignResultsDir)

Reading DORAnet results from : /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/Results/combinedEbolaVirus_bestMACAW_allDB_generative/DORAnet/InoAdeGuoXan/PATHWAY_reconstruction_run2/
DNA design results saved at  : /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/Results/combinedEbolaVirus_bestMACAW_allDB_generative/DNA_design/


## 1. Read list of target DORAnet compounds

In [156]:
targetCompound_DF = pd.read_csv(targetCompoundCsvPath)
targetCompound_DF

,Rank,Canonical_SMILES,pPotency_prediction,pPotency_std,IC50 (µM)
0,1,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,6.156,0.476241,0.697981
1,2,CC(C)(COP(=O)(O)OP(=O)(O)OC[C@H]1O[C@@H](n2cnc...,5.989,0.473052,1.025573
2,3,Cc1cc2c(cc1C)N(C[C@H](O)[C@H](O)[C@H](O)COP(=O...,5.907,0.473610,1.238912
3,4,Cc1cc2nc3c(=O)[nH]c(=O)nc-3n(C[C@H](O)[C@H](O)...,5.825,0.472941,1.495337
4,5,CC(=O)OP(=O)(O)OC(C)=O,5.758,0.479591,1.747229
5,6,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,5.729,0.473567,1.865382
6,7,CC(=O)OP(=O)(O)OP(=O)(O)O,5.693,0.478908,2.026785
7,8,Nc1ncnc2c1ncn2[C@@H]1OC(C(O)O)=C[C@H]1O,5.692,0.473252,2.030622
8,9,CC(Cl)[C@H]1O[C@@H](n2cnc3c(NC=O)ncnc32)[C@H](...,5.683,0.471067,2.074770
9,10,O=C(O)CC(=O)OP(=O)(O)O,5.679,0.476017,2.094051


### Collect DORAnet job configurations

Each DORAnet job directory contains a `reproDoranetJob.py` script recording the starters, target, helpers and generation settings for that job. This cell walks every job folder, parses those values, canonicalizes the SMILES, and joins them to the target-compound potency table — giving one row per DORAnet job.

In [157]:
DORANETmoleculesDataDir = Path(DORANETmoleculesDataDir)

def canonicalizeSmiles(smiles):
    mol = Chem.MolFromSmiles(str(smiles))
    return Chem.MolToSmiles(mol, canonical=True) if mol else None

def canonicalizeSmilesCollection(x):
    if not isinstance(x, (set, list, tuple)):
        return []
    out = []
    for s in x:
        c = canonicalizeSmiles(s)
        if c is not None:
            out.append(c)
    return sorted(set(out))

def parseLiteralFromScript(text, varName):
    # captures lines like: varName = {...} / [...] / "..."
    m = re.search(rf"^\s*{re.escape(varName)}\s*=\s*(.+)\s*$", text, flags=re.MULTILINE)
    if not m:
        return None
    raw = m.group(1).strip()
    try:
        return ast.literal_eval(raw)
    except Exception:
        return None

def parseHelpersFromScript(text):
    # Supports YAML-like line: helpers: ["O", ...]
    m = re.search(r"^\s*helpers\s*:\s*(.+)\s*$", text, flags=re.MULTILINE)
    if not m:
        # fallback in case file uses python assignment: helpers = [...]
        return parseLiteralFromScript(text, "helpers")
    raw = m.group(1).strip()
    try:
        return ast.literal_eval(raw)
    except Exception:
        return None

allReproFiles = sorted(
    DORANETmoleculesDataDir.rglob("reproDoranetJob.py"),
    key=lambda p: str(p).lower()
)

records = []
for reproPath in allReproFiles:
    txt = reproPath.read_text(encoding="utf-8", errors="ignore")

    startersRaw = parseLiteralFromScript(txt, "starters")
    targetRaw = parseLiteralFromScript(txt, "target")
    maxAtomsRaw = parseLiteralFromScript(txt, "maxAtoms")
    generationsRaw = parseLiteralFromScript(txt, "generations")
    helpersRaw = parseHelpersFromScript(txt)

    records.append({
        "dirName": reproPath.parent.name,
        "starters": canonicalizeSmilesCollection(startersRaw),
        "target": canonicalizeSmilesCollection(targetRaw),
        "helpers": canonicalizeSmilesCollection(helpersRaw),   # NEW
        "maxAtoms": maxAtomsRaw if isinstance(maxAtomsRaw, dict) else None,
        "generations": int(generationsRaw) if generationsRaw is not None else None,
    })

targetCompoundDir_DF = pd.DataFrame(
    records,
    columns=["dirName", "starters", "target", "helpers", "maxAtoms", "generations"]  # NEW
)

print(f"Total DORAnet configuration files: {len(targetCompoundDir_DF)}")

# join key from first target smiles
targetCompoundDir_DF = targetCompoundDir_DF.copy()
targetCompoundDir_DF["targetSmiles"] = targetCompoundDir_DF["target"].apply(
    lambda x: x[0] if isinstance(x, (list, tuple)) and len(x) > 0 else None
)

colsToAdd = ["Canonical_SMILES", "pPotency_prediction", "pPotency_std", "IC50 (µM)"]
targetCompoundDir_DF = targetCompoundDir_DF.merge(
    targetCompound_DF[colsToAdd].drop_duplicates(subset=["Canonical_SMILES"]),
    left_on="targetSmiles",
    right_on="Canonical_SMILES",
    how="left"
).drop(columns=["Canonical_SMILES"])

targetCompoundDir_DF = targetCompoundDir_DF.drop(columns=["targetSmiles"])
targetCompoundDir_DF = targetCompoundDir_DF.rename(columns={
    "starters": "starters_Canonical_SMILES",
    "target": "target_Canonical_SMILES",
    "helpers": "helpers_Canonical_SMILES",   
})

for c in ["starters_Canonical_SMILES", "target_Canonical_SMILES"]:
    targetCompoundDir_DF[c] = targetCompoundDir_DF[c].apply(
        lambda x: ".".join(x) if isinstance(x, (list, tuple)) else ("" if pd.isna(x) else str(x))
    )

# Keep helpers as separate entries joined by comma (not dot)
targetCompoundDir_DF["helpers_Canonical_SMILES"] = targetCompoundDir_DF["helpers_Canonical_SMILES"].apply(
    lambda x: ", ".join(x) if isinstance(x, (list, tuple)) else ("" if pd.isna(x) else str(x))
)

targetCompoundDir_DF = targetCompoundDir_DF.sort_values(
    by="pPotency_prediction", ascending=False
).reset_index(drop=True)

# safer path handling
outputPath = Path(DNADesignResultsDir) / "targetCompoundDir_DF.csv"
targetCompoundDir_DF.to_csv(outputPath, index=False)
targetCompoundDir_DF

Total DORAnet configuration files: 60


,dirName,starters_Canonical_SMILES,target_Canonical_SMILES,helpers_Canonical_SMILES,maxAtoms,generations,pPotency_prediction,pPotency_std,IC50 (µM)
0,high_pPotency_molecule_pathway1_wGen1,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,"Br, BrBr, C#N, C=C, C=O, CO, N, N#CO, N#N, NO,...","{'C': 15, 'N': 6, 'O': 8, 'S': 3}",1,6.156,0.476241,0.697981
1,high_pPotency_molecule_pathway1_wGen3,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,"Br, BrBr, C#N, C=C, C=O, CO, N, N#CO, N#N, NO,...","{'C': 15, 'N': 6, 'O': 8, 'S': 3}",3,6.156,0.476241,0.697981
2,high_pPotency_molecule_pathway1_wGen2,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,"Br, BrBr, C#N, C=C, C=O, CO, N, N#CO, N#N, NO,...","{'C': 15, 'N': 6, 'O': 8, 'S': 3}",2,6.156,0.476241,0.697981
3,high_pPotency_molecule_pathway2_wGen1,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C...,CC(C)(COP(=O)(O)OP(=O)(O)OC[C@H]1O[C@@H](n2cnc...,"Br, BrBr, C#N, C=C, C=O, CO, N, N#CO, N#N, NO,...","{'C': 15, 'N': 6, 'O': 8, 'S': 3}",1,5.989,0.473052,1.025573
4,high_pPotency_molecule_pathway2_wGen3,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C...,CC(C)(COP(=O)(O)OP(=O)(O)OC[C@H]1O[C@@H](n2cnc...,"Br, BrBr, C#N, C=C, C=O, CO, N, N#CO, N#N, NO,...","{'C': 15, 'N': 6, 'O': 8, 'S': 3}",3,5.989,0.473052,1.025573
5,high_pPotency_molecule_pathway2_wGen2,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C...,CC(C)(COP(=O)(O)OP(=O)(O)OC[C@H]1O[C@@H](n2cnc...,"Br, BrBr, C#N, C=C, C=O, CO, N, N#CO, N#N, NO,...","{'C': 15, 'N': 6, 'O': 8, 'S': 3}",2,5.989,0.473052,1.025573
6,high_pPotency_molecule_pathway3_wGen2,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C...,Cc1cc2c(cc1C)N(C[C@H](O)[C@H](O)[C@H](O)COP(=O...,"Br, BrBr, C#N, C=C, C=O, CO, N, N#CO, N#N, NO,...","{'C': 15, 'N': 6, 'O': 8, 'S': 3}",2,5.907,0.473610,1.238912
7,high_pPotency_molecule_pathway3_wGen1,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C...,Cc1cc2c(cc1C)N(C[C@H](O)[C@H](O)[C@H](O)COP(=O...,"Br, BrBr, C#N, C=C, C=O, CO, N, N#CO, N#N, NO,...","{'C': 15, 'N': 6, 'O': 8, 'S': 3}",1,5.907,0.473610,1.238912
8,high_pPotency_molecule_pathway3_wGen3,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C...,Cc1cc2c(cc1C)N(C[C@H](O)[C@H](O)[C@H](O)COP(=O...,"Br, BrBr, C#N, C=C, C=O, CO, N, N#CO, N#N, NO,...","{'C': 15, 'N': 6, 'O': 8, 'S': 3}",3,5.907,0.473610,1.238912
9,high_pPotency_molecule_pathway4_wGen3,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C...,Cc1cc2nc3c(=O)[nH]c(=O)nc-3n(C[C@H](O)[C@H](O)...,"Br, BrBr, C#N, C=C, C=O, CO, N, N#CO, N#N, NO,...","{'C': 15, 'N': 6, 'O': 8, 'S': 3}",3,5.825,0.472941,1.495337


## 2. Read the `reconstructed` pathways metadata for all `DORAnet` reactions

- go to the directory where pathway reconstruction DORAnet results are saved
- run this: `python pathwayReconstruction.py config.yaml`
- this will generate the csv file (`doranet_chain_reconstructed_pathway_steps_by_depth.csv`) with reconstructed pathways
- read the csv file for downstream processing

### Load the multi-generation DORAnet job summary

Reads the `config_doranet_multiGen.yaml` for the run and locates its `job_summary.csv`, which lists every DORAnet job to process. The output directory is resolved relative to the config file first, then the working directory, so relative paths keep working across machines.

In [ ]:
doranetMultiGenConfigPath = Path(DORANETmoleculesDataDir) / doranetMultiGenConfigName

with open(doranetMultiGenConfigPath, "r", encoding="utf-8") as f:
    doranetCfg = yaml.safe_load(f) or {}

outputDir = Path(doranetCfg.get("outputDir", ".")).expanduser()

# If outputDir is relative, first try relative to config location.
if not outputDir.is_absolute():
    outputDirFromConfigDir = (doranetMultiGenConfigPath.parent / outputDir).resolve()
    outputDirFromCwd = (Path.cwd() / outputDir).resolve()

    summaryCsvName = doranetCfg.get("summaryCsv", "job_summary.csv")
    if (outputDirFromConfigDir / summaryCsvName).exists():
        outputDir = outputDirFromConfigDir
    else:
        outputDir = outputDirFromCwd

summaryPath = outputDir / doranetCfg.get("summaryCsv", "job_summary.csv")

if not summaryPath.exists():
    raise FileNotFoundError(f"Could not find job summary CSV: {summaryPath}")

jobSummaryDF = pd.read_csv(summaryPath)

print(f"Config file : {doranetMultiGenConfigPath}")
print(f"Output dir  : {outputDir}")
print(f"Summary CSV : {summaryPath}")
print(f"Jobs in summary: {len(jobSummaryDF):,}")

### Parse pathways into a per step reaction table

This is the core parser. For every valid job it reads the `*_pathways.txt` blocks, recovers each reaction's rule name, stoichiometry and enthalpy, and matches reactions back to `*_network_pretreated.json` to recover the reaction type. The result, `reconstructedPathwaysDF`, has **one row per reaction step**, with the pathway (`routeId`) it belongs to.

In [ ]:
def splitMoleculeString(moleculeString):
    if pd.isna(moleculeString):
        return []
    return [mol for mol in str(moleculeString).split(".") if mol]


def sortedMoleculeString(moleculeString):
    mols = splitMoleculeString(moleculeString)
    return ".".join(sorted(mols))


def normalizeReactionSmiles(reactionSmiles):
    """
    Normalize a reaction SMILES by sorting reactant and product molecule strings.

    Input:
        A.B>>C.D

    Output:
        A.B>>C.D with molecules sorted on both sides.
    """
    if pd.isna(reactionSmiles) or ">>" not in str(reactionSmiles):
        return None

    reactants, products = str(reactionSmiles).split(">>", 1)

    reactantsNorm = sortedMoleculeString(reactants)
    productsNorm = sortedMoleculeString(products)

    return f"{reactantsNorm}>>{productsNorm}"


def extractFolderNum(name):
    match = re.search(r"pathway(\d+)", str(name))
    return int(match.group(1)) if match else -1


def splitPathwayBlocks(pathwaysTxtPath):
    if not Path(pathwaysTxtPath).exists():
        return []

    lines = Path(pathwaysTxtPath).read_text(encoding="utf-8").splitlines()

    blocks = []
    current = []

    for line in lines:
        if line.startswith("pathway number ") and current:
            blocks.append(current)
            current = [line]
        else:
            current.append(line)

    if current:
        blocks.append(current)

    blocks = [
        [x for x in block if str(x).strip()]
        for block in blocks
        if any(str(x).strip() for x in block)
    ]

    return blocks


def parsePathwayBlock(block):
    """
    Parse one block from DORAnet *_pathways.txt.

    Expected structure:
        pathway number 1
        place holder ...
        place holder ...
        place holder ...
        reaction SMILES stoichiometry [...]
        reaction SMILES, name, and enthalpy:
        rxn1
        rxn2
        ...
        name1
        name2
        ...
        dH1
        dH2
        ...
    """

    pathwayNumber = None
    for line in block:
        if line.startswith("pathway number "):
            pathwayNumber = str(line).replace("pathway number ", "").strip()
            break

    stoichList = []
    for line in block:
        prefix = "reaction SMILES stoichiometry "
        if line.startswith(prefix):
            payload = line[len(prefix):].strip()
            try:
                stoichList = ast.literal_eval(payload)
            except Exception:
                stoichList = []
            break

    marker = "reaction SMILES, name, and enthalpy:"
    try:
        markerIdx = block.index(marker)
    except ValueError:
        raise ValueError("Could not find reaction SMILES/name/enthalpy marker")

    payloadLines = [x.strip() for x in block[markerIdx + 1:] if str(x).strip()]

    if stoichList:
        numSteps = len(stoichList)
    else:
        if len(payloadLines) % 3 != 0:
            raise ValueError("Cannot infer number of steps from pathway block")
        numSteps = len(payloadLines) // 3
        stoichList = [None] * numSteps

    reactionSmilesList = payloadLines[:numSteps]
    reactionNameList = payloadLines[numSteps:2 * numSteps]
    enthalpyList = payloadLines[2 * numSteps:3 * numSteps]

    if len(reactionSmilesList) != numSteps:
        raise ValueError("Malformed pathway block: reaction count mismatch")

    return {
        "pathwayNumber": pathwayNumber,
        "numSteps": numSteps,
        "reactionSmilesList": reactionSmilesList,
        "reactionNameList": reactionNameList,
        "enthalpyList": enthalpyList,
        "stoichList": stoichList,
    }


def choosePathwayTxt(row):
    """
    Prefer exact-N pathway file generated by filterExactNumRxns.
    Fallback to default *_pathways.txt.
    """

    jobDir = Path(row["jobDir"])
    jobName = str(row["jobName"])
    generationRun = int(row["generationRun"])

    candidatePaths = []

    if "exactPathwaysTxt" in row and pd.notna(row["exactPathwaysTxt"]):
        p = Path(str(row["exactPathwaysTxt"]))
        if not p.is_absolute():
            p = jobDir / p
        candidatePaths.append(p)

    candidatePaths.extend([
        jobDir / f"{jobName}_pathways_exact{generationRun}.txt",
        jobDir / f"{jobName}_pathways.txt",
    ])

    for p in candidatePaths:
        if p.exists():
            return p

    return candidatePaths[0]


def parseReactionSmiles(reactionSmiles):
    if ">>" not in str(reactionSmiles):
        raise ValueError(f"Invalid reaction SMILES: {reactionSmiles}")

    reactants, products = str(reactionSmiles).split(">>", 1)
    return reactants, products


def parseStoich(stoichText):
    if stoichText is None or pd.isna(stoichText):
        return None, None

    parts = str(stoichText).split("$")
    reactantStoich = parts[0] if len(parts) > 0 else None
    productStoich = parts[1] if len(parts) > 1 else None

    return reactantStoich, productStoich


# -------------------------------------------------------------------
# Network JSON lookup functions to recover reactionType
# -------------------------------------------------------------------

def parseNetworkReactionForLookup(rxnString):
    """
    Parse one reaction from *_network_pretreated.json.

    Expected DORAnet format:
        reactants > ruleName > thermo$reactantStoich$productStoich$reactionType > products
    """

    parts = str(rxnString).split(">")

    if len(parts) != 4:
        return None

    reactants, ruleName, metaBlock, products = parts

    metaParts = (metaBlock.split("$") + [None, None, None, None])[:4]
    thermo, reactantStoich, productStoich, reactionType = metaParts

    reactionSmiles = f"{reactants}>>{products}"
    normalizedReactionSmiles = normalizeReactionSmiles(reactionSmiles)

    return {
        "reactionSMILES": reactionSmiles,
        "normalizedReactionSMILES": normalizedReactionSmiles,
        "ruleName": ruleName,
        "thermo": thermo,
        "reactantStoich": reactantStoich,
        "productStoich": productStoich,
        "reactionType": reactionType,
        "rawNetworkReactionString": str(rxnString),
    }


def buildNetworkReactionLookup(networkJsonPath):
    """
    Build lookup dictionaries from *_network_pretreated.json.

    Matching priority later:
        1. normalized reaction SMILES + ruleName
        2. normalized reaction SMILES only
    """

    networkJsonPath = Path(networkJsonPath)

    lookupByReactionAndRule = {}
    lookupByReactionOnly = {}

    if not networkJsonPath.exists():
        return lookupByReactionAndRule, lookupByReactionOnly

    try:
        with open(networkJsonPath, "r", encoding="utf-8") as f:
            reactionList = json.load(f)
    except Exception:
        return lookupByReactionAndRule, lookupByReactionOnly

    if not isinstance(reactionList, list):
        return lookupByReactionAndRule, lookupByReactionOnly

    for rxnString in reactionList:
        parsed = parseNetworkReactionForLookup(rxnString)

        if parsed is None:
            continue

        normRxn = parsed["normalizedReactionSMILES"]
        ruleName = parsed["ruleName"]

        if normRxn is None:
            continue

        lookupByReactionAndRule[(normRxn, ruleName)] = parsed

        if normRxn not in lookupByReactionOnly:
            lookupByReactionOnly[normRxn] = parsed

    return lookupByReactionAndRule, lookupByReactionOnly


def lookupNetworkReactionMetadata(reactionSmiles, ruleName, lookupByReactionAndRule, lookupByReactionOnly):
    """
    Match a pathway reaction back to *_network_pretreated.json.
    """

    normalizedReactionSmiles = normalizeReactionSmiles(reactionSmiles)

    if normalizedReactionSmiles is None:
        return {
            "reactionType": "",
            "rawNetworkReactionString": "",
            "networkMatchStatus": "invalid_reaction_smiles",
        }

    match = lookupByReactionAndRule.get((normalizedReactionSmiles, ruleName))

    if match is not None:
        out = dict(match)
        out["networkMatchStatus"] = "matched_by_reaction_and_rule"
        return out

    match = lookupByReactionOnly.get(normalizedReactionSmiles)

    if match is not None:
        out = dict(match)
        out["networkMatchStatus"] = "matched_by_reaction_only"
        return out

    return {
        "reactionType": "",
        "rawNetworkReactionString": "",
        "networkMatchStatus": "no_network_match",
    }


# -------------------------------------------------------------------
# Build reconstructedPathwaysDF from multi-generation DORAnet outputs
# -------------------------------------------------------------------

starterSet = set(doranetCfg.get("starters", []))
helperSet = set(doranetCfg.get("helpers", []))

records = []
parseErrors = []

networkLookupCache = {}

validJobsDF = jobSummaryDF.copy()

if "status" in validJobsDF.columns:
    validJobsDF = validJobsDF[validJobsDF["status"].eq("ok")].copy()

for _, jobRow in tqdm(validJobsDF.iterrows(), total=len(validJobsDF), desc="Reading DORAnet pathway files"):
    jobName = str(jobRow["jobName"])
    jobDir = Path(jobRow["jobDir"])
    dirName = jobDir.name
    generationRun = int(jobRow["generationRun"])
    targetSMILES = str(jobRow["targetSmiles"])

    pathwayTxtPath = choosePathwayTxt(jobRow)
    networkJsonPath = jobDir / f"{jobName}_network_pretreated.json"

    if jobName not in networkLookupCache:
        networkLookupCache[jobName] = buildNetworkReactionLookup(networkJsonPath)

    lookupByReactionAndRule, lookupByReactionOnly = networkLookupCache[jobName]

    try:
        blocks = splitPathwayBlocks(pathwayTxtPath)

        for blockIdx, block in enumerate(blocks, start=1):
            parsed = parsePathwayBlock(block)

            numSteps = parsed["numSteps"]
            pathwayNumber = parsed["pathwayNumber"] or str(blockIdx)

            routeId = (
                f"{jobName}_route_{int(pathwayNumber):06d}"
                if str(pathwayNumber).isdigit()
                else f"{jobName}_route_{blockIdx:06d}"
            )

            reconstructedPathwayString = "  ||  ".join(parsed["reactionSmilesList"])

            for stepIdx, reactionSmiles in enumerate(parsed["reactionSmilesList"], start=1):
                reactants, products = parseReactionSmiles(reactionSmiles)

                ruleName = (
                    parsed["reactionNameList"][stepIdx - 1]
                    if stepIdx - 1 < len(parsed["reactionNameList"])
                    else ""
                )

                thermoFromPathwayTxt = (
                    parsed["enthalpyList"][stepIdx - 1]
                    if stepIdx - 1 < len(parsed["enthalpyList"])
                    else ""
                )

                stoichText = (
                    parsed["stoichList"][stepIdx - 1]
                    if stepIdx - 1 < len(parsed["stoichList"])
                    else None
                )

                reactantStoichFromTxt, productStoichFromTxt = parseStoich(stoichText)

                networkMatch = lookupNetworkReactionMetadata(
                    reactionSmiles=reactionSmiles,
                    ruleName=ruleName,
                    lookupByReactionAndRule=lookupByReactionAndRule,
                    lookupByReactionOnly=lookupByReactionOnly,
                )

                reactionType = networkMatch.get("reactionType", "")
                rawNetworkReactionString = networkMatch.get("rawNetworkReactionString", "")
                networkMatchStatus = networkMatch.get("networkMatchStatus", "")

                # Prefer pathway file values where available, fallback to network JSON.
                thermo = thermoFromPathwayTxt if thermoFromPathwayTxt not in ["", None] else networkMatch.get("thermo", "")
                reactantStoich = (
                    reactantStoichFromTxt
                    if reactantStoichFromTxt not in ["", None]
                    else networkMatch.get("reactantStoich", None)
                )
                productStoich = (
                    productStoichFromTxt
                    if productStoichFromTxt not in ["", None]
                    else networkMatch.get("productStoich", None)
                )

                reactantMols = set(splitMoleculeString(reactants))
                productMols = set(splitMoleculeString(products))

                record = {
                    "dirName": dirName,
                    "jobName": jobName,
                    "sourceFolderNum": extractFolderNum(jobName),
                    "sourceDirectory": dirName,
                    "sourceDirectoryPath": str(jobDir),
                    "sourceJsonName": f"{jobName}_network_pretreated.json",
                    "sourceJsonPath": str(networkJsonPath),
                    "sourcePathwaysTxtPath": str(pathwayTxtPath),

                    "routeId": routeId,
                    "routeNumberInFile": pathwayNumber,
                    "generationRun": generationRun,
                    "searchDepthUsed": generationRun,
                    "step": stepIdx,
                    "numSteps": numSteps,

                    "starterSMILES": ";".join(sorted(starterSet)),
                    "targetSMILES": targetSMILES,

                    "reactants": reactants,
                    "products": products,
                    "reactionString": f"{reactants} >> {products}",
                    "reactionSMILES": reactionSmiles,
                    "reconstructedPathwayString": reconstructedPathwayString,

                    "ruleName": ruleName,
                    "thermo": thermo,
                    "reactantStoich": reactantStoich,
                    "productStoich": productStoich,
                    "reactionType": reactionType,

                    "rawNetworkReactionString": rawNetworkReactionString,
                    "networkMatchStatus": networkMatchStatus,

                    "numReactantMolecules": len(reactantMols),
                    "numProductMolecules": len(productMols),

                    "starterInReactants": bool(reactantMols & starterSet),
                    "targetInProducts": targetSMILES in productMols,

                    "isReconstructedPathwayStep": True,
                    "routeStepId": f"{routeId}_step_{stepIdx:02d}",
                }

                records.append(record)

    except Exception as exc:
        parseErrors.append({
            "jobName": jobName,
            "jobDir": str(jobDir),
            "generationRun": generationRun,
            "targetSMILES": targetSMILES,
            "pathwayTxtPath": str(pathwayTxtPath),
            "networkJsonPath": str(networkJsonPath),
            "error": str(exc),
        })


reconstructedPathwaysDF = (
    pd.DataFrame(records)
    .sort_values(["sourceFolderNum", "generationRun", "routeId", "step"])
    .reset_index(drop=True)
)

parseErrorsDF = pd.DataFrame(parseErrors)

print(f"Reconstructed pathway step rows : {len(reconstructedPathwaysDF):,}")
print(f"Unique routes                   : {reconstructedPathwaysDF['routeId'].nunique() if not reconstructedPathwaysDF.empty else 0:,}")
print(f"Unique job folders              : {reconstructedPathwaysDF['dirName'].nunique() if not reconstructedPathwaysDF.empty else 0:,}")
print(f"Parse errors                    : {len(parseErrorsDF):,}")

if not reconstructedPathwaysDF.empty:
    print("\nReaction type counts:")
    print(reconstructedPathwaysDF["reactionType"].value_counts(dropna=False).to_string())

    print("\nNetwork match status:")
    print(reconstructedPathwaysDF["networkMatchStatus"].value_counts(dropna=False).to_string())

reconstructedPathwaysDF.head()

### Count starter → target pathways

Rolls the per-step table up to one row per complete route, then pivots to count how many 1-, 2-, and 3-step pathways each starter→target pair has. Useful for seeing which targets are well-covered by the network.

In [ ]:
completeRouteLevelDF = (
    reconstructedPathwaysDF
    .sort_values(["dirName", "routeId", "step"])
    .groupby("routeId", as_index=False)
    .agg(
        dirName=("dirName", "first"),
        jobName=("jobName", "first"),
        sourceFolderNum=("sourceFolderNum", "first"),
        generationRun=("generationRun", "first"),
        numSteps=("numSteps", "first"),
        searchDepthUsed=("searchDepthUsed", "first"),
        starterSMILES=("starterSMILES", "first"),
        targetSMILES=("targetSMILES", "first"),
        reconstructedPathwayString=("reconstructedPathwayString", "first"),
    )
)

starterTargetPathwayCountsDF = (
    completeRouteLevelDF
    .pivot_table(
        index=["dirName", "jobName", "generationRun", "starterSMILES", "targetSMILES"],
        columns="numSteps",
        values="routeId",
        aggfunc="nunique",
        fill_value=0,
    )
    .rename(columns={
        1: "numOneStepPathways",
        2: "numTwoStepPathways",
        3: "numThreeStepPathways",
    })
    .reset_index()
)

for c in ["numOneStepPathways", "numTwoStepPathways", "numThreeStepPathways"]:
    if c not in starterTargetPathwayCountsDF.columns:
        starterTargetPathwayCountsDF[c] = 0

starterTargetPathwayCountsDF["totalStarterToTargetPathways"] = (
    starterTargetPathwayCountsDF["numOneStepPathways"]
    + starterTargetPathwayCountsDF["numTwoStepPathways"]
    + starterTargetPathwayCountsDF["numThreeStepPathways"]
)

starterTargetPathwayCountsDF = (
    starterTargetPathwayCountsDF
    .sort_values("totalStarterToTargetPathways", ascending=False)
    .reset_index(drop=True)
)

print(f"Total reaction-step rows             : {len(reconstructedPathwaysDF):,}")
print(f"Complete starter --> target pathways: {len(completeRouteLevelDF):,}")
print(f"Job folders reached starter --> target: {starterTargetPathwayCountsDF['dirName'].nunique():,}")

starterTargetPathwayCountsDF

### `Thermodynamic (ΔH)` feasibility check

Estimates each reaction's enthalpy change using the Joback group-contribution method (with hardcoded gas-phase values for small inorganics that Joback cannot fragment). Routes are then flagged `thermoFeasible` if **every** step falls under the `maxRxnThermoChange` cutoff — mirroring DORAnet's "a route is only as good as its worst step" logic. Tune `maxRxnThermoChange` to your own ΔH distribution.

In [ ]:
from thermo.group_contribution.joback import Joback

hfCacheKcalPerMol: dict[str, float | None] = {}

from rdkit import Chem

# Gas-phase ΔfH° (kcal/mol), small inorganics Joback's organic-only group scheme
# can't fragment at all (no carbon backbone). Sourced from NIST WebBook / CODATA
# unless noted. H2 and N2 are zero by definition (reference elemental states).
# Confidence is high for H2O/NH3/CO/H2/N2/CO2; lower for the sulfur/nitrogen
# oxoacids, since gas-phase data for those is less commonly tabulated -- verify
# against NIST WebBook directly if your network leans heavily on them.
knownHfKcalPerMol = {
    "O":          -57.80,   # H2O, NIST/CODATA gas phase
    "N":          -11.02,   # NH3, NIST/CODATA gas phase
    "S":           -4.93,   # H2S, NIST/CODATA gas phase
    "[H][H]":       0.00,   # H2, reference element
    "N#N":          0.00,   # N2, reference element
    "C=O":        -25.95,   # CH2O (formaldehyde), gas phase -- moderate confidence
    "[C-]#[O+]":  -26.42,   # CO, NIST/CODATA gas phase
    "O=[N+]([O-])O": -32.10,  # HNO3, gas phase -- lower confidence, verify
    "O=S(=O)(O)O":   None,    # H2SO4, gas-phase data too uncertain to assert -- left as None
    "O=S(O)O":       None,    # H2SO3, same caveat
}

def calculateHfCached(smiles: str) -> float | None:
    if smiles not in hfCacheKcalPerMol:
        canon = Chem.MolToSmiles(Chem.MolFromSmiles(smiles)) if Chem.MolFromSmiles(smiles) else smiles
        if canon in knownHfKcalPerMol:
            hfCacheKcalPerMol[smiles] = knownHfKcalPerMol[canon]
        else:
            try:
                j = Joback(smiles)
                hfCacheKcalPerMol[smiles] = j.Hf(j.counts) / 4184 if j.status == "OK" else None
            except Exception:
                hfCacheKcalPerMol[smiles] = None
    return hfCacheKcalPerMol[smiles]


def parseStoichTuple(stoichText):
    if stoichText is None or pd.isna(stoichText):
        return None
    try:
        parsed = ast.literal_eval(str(stoichText))
        return tuple(parsed) if isinstance(parsed, (list, tuple)) else None
    except Exception:
        return None


def computeStepDH(row) -> tuple[float | None, str, bool]:
    reactantMols = splitMoleculeString(row["reactants"])
    productMols = splitMoleculeString(row["products"])

    reactantStoich = parseStoichTuple(row.get("reactantStoich"))
    productStoich = parseStoichTuple(row.get("productStoich"))

    stoichWasInferred = False
    if reactantStoich is None or len(reactantStoich) != len(reactantMols):
        reactantStoich = (1,) * len(reactantMols)
        stoichWasInferred = True
    if productStoich is None or len(productStoich) != len(productMols):
        productStoich = (1,) * len(productMols)
        stoichWasInferred = True

    reactantHfs = [calculateHfCached(s) for s in reactantMols]
    productHfs = [calculateHfCached(s) for s in productMols]

    if any(hf is None for hf in reactantHfs + productHfs):
        return None, "missing_hf", stoichWasInferred

    dH = sum(hf * n for hf, n in zip(productHfs, productStoich)) \
       - sum(hf * n for hf, n in zip(reactantHfs, reactantStoich))
    return round(dH, 4), "ok", stoichWasInferred


dHResults = [
    computeStepDH(row)
    for _, row in tqdm(reconstructedPathwaysDF.iterrows(),
                        total=len(reconstructedPathwaysDF),
                        desc="Computing reaction enthalpies (Joback)")
]

reconstructedPathwaysDF["dH_kcal_per_mol"] = [r[0] for r in dHResults]
reconstructedPathwaysDF["dHStatus"] = [r[1] for r in dHResults]
reconstructedPathwaysDF["stoichWasInferred"] = [r[2] for r in dHResults]

print(reconstructedPathwaysDF["dHStatus"].value_counts(dropna=False).to_string())
print(f"Steps with inferred (1,1,...) stoichiometry: {reconstructedPathwaysDF['stoichWasInferred'].sum():,}")

# -------------------------------------------------------------------
# Roll step-level dH up to the route level, same logic DORAnet's own
# pathway_ranking() uses: a route is only as good as its worst step.
# -------------------------------------------------------------------
maxRxnThermoChange = 15  # kcal/mol -- tune this against your own dH distribution

routeThermoDF = (
    reconstructedPathwaysDF
    .groupby("routeId", as_index=False)
    .agg(
        maxStepDH=("dH_kcal_per_mol", "max"),
        minStepDH=("dH_kcal_per_mol", "min"),
        numStepsWithThermo=("dH_kcal_per_mol", lambda s: s.notna().sum()),
        numStepsTotal=("dH_kcal_per_mol", "size"),
        anyStoichInferred=("stoichWasInferred", "any"),
    )
)
routeThermoDF["allStepsHaveThermo"] = routeThermoDF["numStepsWithThermo"] == routeThermoDF["numStepsTotal"]
routeThermoDF["thermoFeasible"] = routeThermoDF["allStepsHaveThermo"] & (routeThermoDF["maxStepDH"] < maxRxnThermoChange)

completeRouteLevelDF = completeRouteLevelDF.merge(routeThermoDF, on="routeId", how="left")
print(f"\nRoutes with full thermo coverage: {routeThermoDF['allStepsHaveThermo'].sum():,} / {len(routeThermoDF):,}")
print(f"Routes passing {maxRxnThermoChange} kcal/mol cutoff: {routeThermoDF['thermoFeasible'].sum():,}")

### Keep only `enzymatic` DORAnet reactions

Filters `reconstructedPathwaysDF` to reactions whose type looks enzymatic/biological, since only those are candidates for enzyme assignment and DNA design.

In [ ]:
enzymaticReactionDF = reconstructedPathwaysDF[
    reconstructedPathwaysDF["reactionType"].astype(str).str.lower().str.contains(
        "enzyme|enzymatic|bio|biological",
        na=False
    )
].copy()

print(f"Total reactions: {len(reconstructedPathwaysDF):,}")
print(f"Likely enzymatic reactions: {len(enzymaticReactionDF):,}")

enzymaticReactionDF.head()

## 2. Use `DORA-XGB` to get feasibility score

Applies all four DORA-XGB feasibility classifiers to each reaction string, producing a probability score and a 0/1 label per rule. Uncomment the `.head(50)` line while developing to run on a small sample first.

In [ ]:
reactionDF_DORAXGB = enzymaticReactionDF.copy()
#reactionDF_DORAXGB = enzymaticReactionDF.head(50).copy()

# Clean reaction string for model input
reactionDF_DORAXGB["rxn_str"] = (
    reactionDF_DORAXGB["reactionString"]
    .astype(str)
    .str.replace(" ", "", regex=False)
)

def getFeasibilityScoresAndLabels(rxnStr):
    return pd.Series({
        "feasibilityScore_rule1": by_desc_MW_model.predict_proba(rxnStr),
        "feasibilityLabel_rule1": by_desc_MW_model.predict_label(rxnStr),

        "feasibilityScore_rule2": by_asc_MW_model.predict_proba(rxnStr),
        "feasibilityLabel_rule2": by_asc_MW_model.predict_label(rxnStr),

        "feasibilityScore_rule3": add_concat_model.predict_proba(rxnStr),
        "feasibilityLabel_rule3": add_concat_model.predict_label(rxnStr),

        "feasibilityScore_rule4": add_subtract_model.predict_proba(rxnStr),
        "feasibilityLabel_rule4": add_subtract_model.predict_label(rxnStr),
    })


reactionDF_DORAXGB["rxn_str"] = (
    reactionDF_DORAXGB["reactionString"]
    .astype(str)
    .str.replace(" ", "", regex=False)
)

reactionDF_DORAXGB[
    [
        "feasibilityScore_rule1",
        "feasibilityLabel_rule1",
        "feasibilityScore_rule2",
        "feasibilityLabel_rule2",
        "feasibilityScore_rule3",
        "feasibilityLabel_rule3",
        "feasibilityScore_rule4",
        "feasibilityLabel_rule4",
    ]
] = reactionDF_DORAXGB["rxn_str"].apply(getFeasibilityScoresAndLabels)

reactionDF_DORAXGB.head()

### Keep only `high feasible` reactions

Pick which of the four DORA-XGB labels to treat as the decision (`feasibilityLabel`) and which label value counts as feasible (`targetValue`). The safety checks below fail fast if you name a column that does not exist.

In [ ]:
# Choose which feasibility label to use
feasibilityLabel = "feasibilityLabel_rule2"   # <- change to rule2/rule3/rule4 as needed
targetValue = 1                               # <- change if you want a different label value

# Optional safety check
validLabels = [
    "feasibilityLabel_rule1",
    "feasibilityLabel_rule2",
    "feasibilityLabel_rule3",
    "feasibilityLabel_rule4",
]
if feasibilityLabel not in validLabels:
    raise ValueError(f"Invalid feasibilityLabel: {feasibilityLabel}. Choose from {validLabels}")

if feasibilityLabel not in reactionDF_DORAXGB.columns:
    raise KeyError(f"Column '{feasibilityLabel}' not found in reactionDF_DORAXGB")

### Keep only high-feasibility reactions

Applies the chosen feasibility filter, summarizes the unique reactants/products, and saves the surviving reactions to CSV in the DNA-design output folder.

In [ ]:
reactionDF_DORAXGB_highFeasibility = (
    reactionDF_DORAXGB[
        (reactionDF_DORAXGB[feasibilityLabel] == targetValue)
        & (reactionDF_DORAXGB[feasibilityLabel].notna())
    ]
    .sort_values(by=feasibilityLabel, ascending=False)
    .reset_index(drop=True)
)

uniqueReactantStrings_high = set(reactionDF_DORAXGB_highFeasibility["reactants"])
uniqueProductStrings_high = set(reactionDF_DORAXGB_highFeasibility["products"])

uniqueReactantMolecules_high = {
    mol.strip()
    for reactants in reactionDF_DORAXGB_highFeasibility["reactants"]
    for mol in str(reactants).split(".")
    if mol.strip()
}

uniqueProductMolecules_high = {
    mol.strip()
    for products in reactionDF_DORAXGB_highFeasibility["products"]
    for mol in str(products).split(".")
    if mol.strip()
}

print(f"Feasibility label used: {feasibilityLabel} == {targetValue}")
print(f"Number of high-feasibility reactions: {len(reactionDF_DORAXGB_highFeasibility)}")
print(f"Number of unique reactant strings: {len(uniqueReactantStrings_high)}")
print(f"Number of unique product strings: {len(uniqueProductStrings_high)}")
print(f"Number of unique individual reactant molecules: {len(uniqueReactantMolecules_high)}")
print(f"Number of unique individual product molecules: {len(uniqueProductMolecules_high)}")

reactionDF_DORAXGB_highFeasibility.to_csv(os.path.join(DNADesignResultsDir, f"DORAXGB_highFeasibility_{feasibilityLabel}.csv"),index=False)
reactionDF_DORAXGB_highFeasibility.head()

### Summarize unique DORAnet rules

Many reactions share the same DORAnet rule. Rather than annotating millions of reactions individually, we group by rule so enzyme assignment happens once per rule.

In [ ]:
enzymaticReactionDF = reactionDF_DORAXGB_highFeasibility.copy()
ruleSummaryDF = (
    enzymaticReactionDF
    .groupby(["ruleName", "reactionType"], dropna=False)
    .agg(
        numReactions=("reactionString", "count"),
        #numStarterSources=("sourceFolderNum", "nunique"),
        exampleReaction=("reactionString", "first"),
        exampleReactants=("reactants", "first"),
        exampleProducts=("products", "first"),
    )
    .reset_index()
    .sort_values("numReactions", ascending=False)
)

print(f"unique DORAnet rules    : {enzymaticReactionDF['ruleName'].nunique():,}")
ruleSummaryDF

### Add placeholder annotation columns

Adds empty columns (EC number, enzyme name, UniProt accession, confidence, notes) that later steps.

In [ ]:
enzymeAnnotationDF = ruleSummaryDF.copy()

enzymeAnnotationDF["suggestedEnzymeClass"] = ""
enzymeAnnotationDF["ecNumber"] = ""
enzymeAnnotationDF["enzymeName"] = ""
enzymeAnnotationDF["uniprotAccession"] = ""
enzymeAnnotationDF["sourceDatabase"] = ""
enzymeAnnotationDF["reactionSimilarity"] = ""
enzymeAnnotationDF["enzymeConfidence"] = ""
enzymeAnnotationDF["notes"] = ""
enzymeAnnotationDF.head()

## 3. Read DORAnet reaction ruleset file to extract `UniProt ID` for each `reaction rule`

### Load the DORAnet rule set

Reads the DORAnet rule TSV (`JN3604IMT_rules.tsv` by default), which maps each rule to its SMARTS and candidate UniProt IDs.

In [ ]:
doranetRulesetDF = pd.read_csv(doranetRulesetPath, sep="\t")

print(doranetRulesetDF.shape)
print(doranetRulesetDF.columns.tolist())
doranetRulesetDF.head()

### Find the number of common reaction rules between `DORAnet` diversification and ruleset table from actual package

**Compare rules used vs. rules available:** Checks how many rules from this run are present in the packaged rule set, so you know how many reactions can be looked up directly.

In [ ]:
reactionRuleSet = set(reactionDF_DORAXGB_highFeasibility["ruleName"].dropna().astype(str).str.strip())
tsvRuleSet = set(doranetRulesetDF["Name"].dropna().astype(str).str.strip())

matchedRuleSet = reactionRuleSet.intersection(tsvRuleSet)
missingRuleSet = reactionRuleSet.difference(tsvRuleSet)

print(f"Rules present in DORAnet diversification run: {len(reactionRuleSet):,}")
print(f"Rules matched from DORAnet package: {len(matchedRuleSet):,}")
print(f"Rules missing from DORAnet package: {len(missingRuleSet):,}")

print("\nMatched rules:")
print(sorted(list(matchedRuleSet))[:20])

print("\nMissing rules:")
print(sorted(list(missingRuleSet))[:20])

### Extract `UniProt IDs` from `DORAnet` reaction rules

Renames the rule-set columns to consistent names and splits the semicolon-delimited UniProt IDs in the `Comments` field into a clean list per rule.

In [ ]:
def splitUniProtIds(idString):
    if pd.isna(idString):
        return []
    return [x.strip() for x in str(idString).split(";") if x.strip()]


ruleInfoDF = doranetRulesetDF.copy()

ruleInfoDF = ruleInfoDF.rename(columns={
    "Name": "ruleName",
    "Reactants": "ruleReactants",
    "SMARTS": "ruleSMARTS",
    "Products": "ruleProducts",
    "Comments": "candidateUniProtRaw",
})

ruleInfoDF["ruleName"] = ruleInfoDF["ruleName"].astype(str).str.strip()
ruleInfoDF["candidateUniProtList"] = ruleInfoDF["candidateUniProtRaw"].apply(splitUniProtIds)
ruleInfoDF["numCandidateUniProt"] = ruleInfoDF["candidateUniProtList"].apply(len)

ruleInfoDF = ruleInfoDF[
    [
        "ruleName",
        "ruleReactants",
        "ruleProducts",
        "ruleSMARTS",
        "candidateUniProtRaw",
        "numCandidateUniProt",
    ]
].copy()



print(f"ruleInfoDF rows: {len(ruleInfoDF):,}")
ruleInfoDF

## 4. Merge rule information into `reactionDF`

Joins each reaction to its rule SMARTS and candidate UniProt list, and flags whether a rule lookup succeeded (`hasRuleLookup`).

In [ ]:
reactionDF_wUniprotID = reactionDF_DORAXGB_highFeasibility.merge(ruleInfoDF,on="ruleName",how="left")

reactionDF_wUniprotID["hasRuleLookup"] = reactionDF_wUniprotID["ruleSMARTS"].notna()

print(reactionDF_wUniprotID["hasRuleLookup"].value_counts(dropna=False))
reactionDF_wUniprotID

## 4. Find best `UniProt ID` per rule by `Reaction Center Morgan Fingerprint (RCMFP)` enzyme retrieval similarity search 

Implements workflow from the DORAnet/DORA-XGB → atom mapping → RCMFP → known enzyme retrieval. It uses `ruleBase` as the practical bridge between expanded DORAnet rule names such as `rule0003_170` and known reference coarse operator IDs such as `rule0003`, then ranks candidate natural enzyme precedents by RCMFP Tanimoto similarity.

Use this tool: https://github.com/stefanpate/ergochemics/tree/main#

### Patch `RDKit / ergochemics` compatibility

Shims `MolToSmiles` so it tolerates the `ignoreAtomMapNumbers` keyword across RDKit versions **while preserving atom-map labels** — RCMFP needs those maps to locate reaction centers, so do not use a patch that strips them.

In [ ]:
# ERGOCHEMICS_SRC is defined in the CONFIG cell.
if ERGOCHEMICS_SRC not in sys.path: sys.path.insert(0, ERGOCHEMICS_SRC)

import ergochemics.mapping as egmap
import ergochemics.similarity as egsim
print("RDKit version:", rdBase.rdkitVersion); print("ergochemics mapping file:", egmap.__file__)

_RDKit_MolToSmiles_original = rdmolfiles.MolToSmiles

def MolToSmiles_compat(mol, *args, **kwargs):
    kwargs.pop("ignoreAtomMapNumbers", None)
    return _RDKit_MolToSmiles_original(mol, *args, **kwargs)

Chem.MolToSmiles = MolToSmiles_compat; egmap.Chem.MolToSmiles = MolToSmiles_compat; egsim.Chem.MolToSmiles = MolToSmiles_compat
operator_map_reaction, get_reaction_center = egmap.operator_map_reaction, egmap.get_reaction_center
ReactionFingerprinter, MolFeaturizer = egsim.ReactionFingerprinter, egsim.MolFeaturizer
print("Patch test:", Chem.MolToSmiles(Chem.MolFromSmiles("[CH3:1][OH:2]"), ignoreAtomMapNumbers=True))

### Normalize and atom-map every reaction

Cleans reaction strings and atom-maps each one against its rule SMARTS (trying both implicit and explicit-hydrogen modes). Reactions that map successfully move forward; the rest are kept separately for inspection.

In [ ]:
def normalize_reaction_string(rxn):
    if pd.isna(rxn): return None
    rxn = str(rxn).strip().replace(" ", "")
    if rxn.count(">") == 2 and ">>" in rxn: return rxn
    parts = rxn.split(">")
    return f"{parts[0]}>>{parts[2]}" if len(parts) == 3 else None

def map_query_reaction_with_rule(row):
    rxn, ruleSMARTS = row.get("queryReaction"), row.get("ruleSMARTS")
    if pd.isna(rxn) or str(rxn).strip() == "": return None, "missing_queryReaction"
    if pd.isna(ruleSMARTS) or str(ruleSMARTS).strip() == "": return None, "missing_ruleSMARTS"
    rxn, ruleSMARTS, lastError = str(rxn).replace(" ", ""), str(ruleSMARTS).strip(), None
    if ">>" not in rxn: return None, "invalid_queryReaction_no_double_arrow"
    if ">>" not in ruleSMARTS: return None, "invalid_ruleSMARTS_no_double_arrow"
    for explicitHsFlag, statusLabel in [(False, "mapped"), (True, "mapped_explicit_h")]:
        try:
            result = operator_map_reaction(rxn=rxn, operator=ruleSMARTS, explicit_hs=explicitHsFlag, quiet=True)
            if result.did_map and result.atom_mapped_smarts is not None: return result.atom_mapped_smarts, statusLabel
            lastError = f"{statusLabel}_failed"
        except Exception as exc: lastError = f"mapping_error:{type(exc).__name__}:{exc}"
    return None, lastError

AtomMapDF = reactionDF_wUniprotID.copy()
AtomMapDF["queryReaction"] = AtomMapDF["reactionString"].apply(normalize_reaction_string)
AtomMapDF = AtomMapDF[AtomMapDF["queryReaction"].notna() & AtomMapDF["ruleSMARTS"].notna()].drop(columns=["queryMappedReaction", "rcmfpMappingStatus"], errors="ignore").copy()

mappedPairs = AtomMapDF.progress_apply(map_query_reaction_with_rule, axis=1)
AtomMapDF[["queryMappedReaction", "rcmfpMappingStatus"]] = pd.DataFrame(mappedPairs.tolist(), index=AtomMapDF.index)
sucessAtomMapDF = AtomMapDF[AtomMapDF["queryMappedReaction"].notna()].copy() 
failedAtomMapDF = AtomMapDF[AtomMapDF["queryMappedReaction"].isna()].copy()
print(AtomMapDF["rcmfpMappingStatus"].value_counts(dropna=False))
print("Successful atom mapping:", len(sucessAtomMapDF), "| Failed:", len(failedAtomMapDF))
sucessAtomMapDF.shape

### RCMFP fingerprint parameters

`TOP_K` matches to keep per query, minimum similarity, and the Morgan fingerprint size/radius.

In [ ]:
TOP_K, MIN_SIMILARITY, FP_SIDE_LENGTH, FP_RADIUS = 20, 0.0, 2048, 2

### Compute query RCMFP fingerprints

Builds a Reaction-Center Morgan Fingerprint for each successfully atom-mapped reaction, recording a status so failures (empty reaction center, missing atom maps, etc.) are traceable.

In [ ]:
reactionFingerprinter = ReactionFingerprinter(radius=FP_RADIUS, length=FP_SIDE_LENGTH, mol_featurizer=MolFeaturizer())

def compute_rcmfp_with_status(mappedRxn):
    try:
        if pd.isna(mappedRxn) or str(mappedRxn).strip() == "": return None, "missing_mapped_reaction"
        mappedRxn = str(mappedRxn).replace(" ", "")
        if ">>" not in mappedRxn: return None, "invalid_mapped_reaction_no_double_arrow"
        if not pd.Series([mappedRxn]).str.contains(r":\d+\]", regex=True).iloc[0]: return None, "no_atom_map_labels"
        lrc, rrc = get_reaction_center(mappedRxn, mode="combined")
        if len(lrc) == 0 or len(rrc) == 0: return None, f"empty_reaction_center:lrc={len(lrc)}:rrc={len(rrc)}"
        fp = reactionFingerprinter.fingerprint(mappedRxn, output_type="bit", use_rc=True, rc_dist_ub=None)
        return fp.astype(bool), "rcmfp_success"
    except Exception as exc: return None, f"rcmfp_error:{type(exc).__name__}:{exc}"


rcmfpPairs = sucessAtomMapDF["queryMappedReaction"].progress_apply(compute_rcmfp_with_status)
sucessAtomMapDF[["queryRCMFP", "rcmfpStatus"]] = pd.DataFrame(rcmfpPairs.tolist(), index=sucessAtomMapDF.index)
queryRcmfpFailureDF = sucessAtomMapDF[sucessAtomMapDF["queryRCMFP"].isna()].copy()
sucessAtomMapDF = sucessAtomMapDF[sucessAtomMapDF["queryRCMFP"].notna()].copy().reset_index(drop=True)
sucessAtomMapDF

In [ ]:
print("RCMFP status counts:")
print(sucessAtomMapDF["rcmfpStatus"].value_counts(dropna=False))
print("Valid RCMFP reactions:", len(sucessAtomMapDF))
print("Failed RCMFP reactions:", len(queryRcmfpFailureDF))

### Load and align known enzyme reference metadata and RCMFP cache

- loaded from: https://github.com/JBEI/TridentSynthWeb/tree/b17e61040b182ce68b2931f8dbbaf65f370f8c03/data/processed

### Load the known-enzyme reference and its RCMFP cache

Loads the reference set of known natural enzyme reactions plus their precomputed fingerprint matrix, and aligns the two by the cached valid indices.

In [ ]:
naturalEnzymeFilePath    = knownEnzymeParquetPath
naturalEnzyme_FP_FilePath = knownEnzymeRcmfpPath

naturalEnzymeDF = pd.read_parquet(naturalEnzymeFilePath)
naturalEnzymeDF_wFingerprints = np.load(naturalEnzyme_FP_FilePath, allow_pickle=True)
naturalEnzyme_fingerprint_matrix = naturalEnzymeDF_wFingerprints["fingerprints"].astype(bool)
naturalEnzyme_RC_PatternLHS, naturalEnzyme_RC_PatternRHS, knownValidIndices = naturalEnzymeDF_wFingerprints["rc_patterns_lhs"], naturalEnzymeDF_wFingerprints["rc_patterns_rhs"], naturalEnzymeDF_wFingerprints["valid_indices"]
naturalEnzymeReferenceDF = naturalEnzymeDF.iloc[knownValidIndices].reset_index(drop=True).copy()
naturalEnzymeReferenceDF["naturalEnzyme_RC_PatternLHS"], naturalEnzymeReferenceDF["naturalEnzyme_RC_PatternRHS"], naturalEnzymeReferenceDF["knownOriginalIndex"] = naturalEnzyme_RC_PatternLHS, naturalEnzyme_RC_PatternRHS, knownValidIndices
assert len(naturalEnzymeReferenceDF) == naturalEnzyme_fingerprint_matrix.shape[0]
print("Known enzyme reference shape:", naturalEnzymeReferenceDF.shape); print("Known enzyme molecular fingerprint matrix shape:", naturalEnzyme_fingerprint_matrix.shape)

### Compute RCMFP fingerprint matrices for `DORAnet` reactions

Stacks the query and reference fingerprints into boolean matrices (plus reversed copies, so a reaction can match a reference in either direction) and precomputes popcounts for fast Tanimoto scoring.

In [ ]:
def make_reverse_fingerprint_matrix(fpMatrix, sideLength=FP_SIDE_LENGTH): return np.hstack([fpMatrix[:, sideLength:], fpMatrix[:, :sideLength]]).astype(bool)
def make_fingerprint_matrix_from_column(df, fpCol):
    validMask = df[fpCol].notna(); metaDF = df.loc[validMask].drop(columns=[fpCol], errors="ignore").reset_index(drop=True)
    fpMatrix = np.vstack(df.loc[validMask, fpCol].to_numpy()).astype(bool); fpReverseMatrix = make_reverse_fingerprint_matrix(fpMatrix)
    return metaDF, fpMatrix, fpReverseMatrix, fpMatrix.sum(axis=1), fpReverseMatrix.sum(axis=1)

queryMetaDF, query_FP_Matrix, query_FP_MatrixReverse, queryPopcounts, queryReversePopcounts = make_fingerprint_matrix_from_column(sucessAtomMapDF, "queryRCMFP")
naturalEnzyme_fingerprint_matrixReverse = make_reverse_fingerprint_matrix(naturalEnzyme_fingerprint_matrix)
knownPopcounts, knownReversePopcounts = naturalEnzyme_fingerprint_matrix.sum(axis=1), naturalEnzyme_fingerprint_matrixReverse.sum(axis=1)

queryKeepCols = ["reactants", "products", "reactionString", "ruleName", "reactionType", "rxn_str", "ruleSMARTS", "candidateUniProtRaw", "numCandidateUniProt", "hasRuleLookup", "queryReaction", "queryMappedReaction", "rcmfpMappingStatus", "rcmfpStatus", "feasibilityScore_rule1", "feasibilityLabel_rule1", "feasibilityScore_rule2", "feasibilityLabel_rule2", "feasibilityScore_rule3", "feasibilityLabel_rule3", "feasibilityScore_rule4", "feasibilityLabel_rule4"]
queryMetaDF = queryMetaDF[[c for c in queryKeepCols if c in queryMetaDF.columns]].copy()
print("Query metadata shape:", queryMetaDF.shape); print("Query molecular fingerprint matrix shape:", query_FP_Matrix.shape); print("Known enzyme molecular fingerprint matrix shape:", naturalEnzyme_fingerprint_matrix.shape)
queryMetaDF

### Add `ruleBase` column to known enzyme reference and query metadata from DORAnet reactions

Reduces expanded rule names (e.g. `rule0003_170`) to coarse operator families (e.g. `rule0003`). Retrieval is then constrained within the same family, letting RCMFP pick the closest natural reaction inside it.

In [ ]:
def get_rule_base(x): return None if pd.isna(x) else str(x).strip().split("_")[0]
def parse_operator_list(x):
    if isinstance(x, np.ndarray): return [str(v).strip() for v in x.tolist() if v is not None and str(v).strip()]
    if isinstance(x, (list, tuple, set)): return [str(v).strip() for v in list(x) if v is not None and str(v).strip()]
    if x is None or pd.isna(x): return []
    s = str(x).strip()
    if s in ["", "nan", "None", "[]"]: return []
    try:
        parsed = ast.literal_eval(s)
        return [str(v).strip() for v in list(parsed)] if isinstance(parsed, (list, tuple, set, np.ndarray)) else [str(parsed).strip()]
    except Exception:
        for sep in ["|", ";", ","]:
            if sep in s: return [v.strip().strip("'\"") for v in s.split(sep) if v.strip()]
        return [s]

def choose_known_rule_base(row):
    top = row.get("top_mapped_operator")
    if top is not None and not pd.isna(top) and str(top).strip().startswith("rule"): return str(top).strip(), "top_mapped_operator"
    for op in parse_operator_list(row.get("all_mapped_operators")):
        if str(op).startswith("rule"): return str(op), "all_mapped_operators"
    return None, "no_ruleBase"



# Build ruleBase in naturalEnzymeReferenceDF
naturalEnzymeReferenceDF = naturalEnzymeReferenceDF.copy()

naturalEnzymeReferenceDF[["ruleBase", "ruleBaseSource"]] = naturalEnzymeReferenceDF.apply(
    lambda row: pd.Series(choose_known_rule_base(row)),
    axis=1
)

# If you want only base (e.g., rule0001 from rule0001_22), normalize:
naturalEnzymeReferenceDF["ruleBase"] = naturalEnzymeReferenceDF["ruleBase"].apply(get_rule_base)

# Rebuild metadata
queryMetaDF = queryMetaDF.reset_index(drop=True).copy()
knownMetaDF = naturalEnzymeReferenceDF.reset_index(drop=True).copy()

queryMetaDF["queryRowId"] = np.arange(len(queryMetaDF))
knownMetaDF["knownRowId"] = np.arange(len(knownMetaDF))

queryMetaDF["ruleBase"] = queryMetaDF["ruleName"].apply(get_rule_base)

queryMetaDF["_ruleKey"] = queryMetaDF["ruleBase"].astype("object")
knownMetaDF["_ruleKey"] = knownMetaDF["ruleBase"].astype("object")

# Create ruleBase first, then use it
queryMetaDF["ruleBase"] = queryMetaDF["ruleName"].apply(get_rule_base)

if "ruleBase" not in knownMetaDF.columns:
    raise KeyError("knownMetaDF does not contain ruleBase after merge. Check knownOriginalIndex alignment.")

queryMetaDF["_ruleKey"] = queryMetaDF["ruleBase"].astype("object")
knownMetaDF["_ruleKey"] = knownMetaDF["ruleBase"].astype("object")

knownIndicesByRuleBase = {
    ruleBase: idx.to_numpy()
    for ruleBase, idx in knownMetaDF.dropna(subset=["_ruleKey"]).groupby("_ruleKey").groups.items()
}

queryRuleBases = set(queryMetaDF["ruleBase"].dropna().astype(str))
knownRuleBases = set(knownMetaDF["ruleBase"].dropna().astype(str))

print("Source of known enzymetic data base:")
print(knownMetaDF["ruleBaseSource"].value_counts(dropna=False))
print("Reaction rules present in DORAnet reaction mechanism:", len(queryRuleBases))
print("Reaction rules present in known enzymetic data base:", len(knownRuleBases))
print("Overlapping base reaction rules:", len(queryRuleBases.intersection(knownRuleBases)))
print("Example overlaps:", sorted(queryRuleBases.intersection(knownRuleBases))[:20])

### Run `ruleBase` constrained RCMFP enzyme retrieval

For each query reaction (present in DORAnet reaction network) it searches the known natural enzyme reaction database, computes RCMFP Tanimoto similarity, keeps the best matches and stores the results in `matchedRCMFP_DF`. Final returned matches are ranked from highest to lowest RCMFP similarity.

In [ ]:
# this function calculates Tanimoto similarity between one query fingerprint and many known enzyme reference fingerprints
def compute_tanimoto_similarity(queryPackedRow, queryPop, refPackedMatrix, refPopcounts, popcount8):
    inter = popcount8[np.bitwise_and(refPackedMatrix, queryPackedRow)].sum(axis=1).astype(np.float32); union = refPopcounts + queryPop - inter
    scores = np.zeros(len(refPopcounts), dtype=np.float32); valid = union > 0; scores[valid] = inter[valid] / union[valid]
    return scores

# this function retrieves the top enzyme reference matches for one query reaction
def retrieve_similar_reactions(queryIdx, topK=TOP_K, minSimilarity=MIN_SIMILARITY):
    queryRuleBase = queryMetaDF.loc[queryIdx, "_ruleKey"]
    candidateIdx, searchMode = (knownIndicesByRuleBase[queryRuleBase], "same_ruleBase") if pd.notna(queryRuleBase) and queryRuleBase in knownIndicesByRuleBase else (np.arange(len(knownMetaDF)), "full_database_fallback")
    qPacked, qPop = queryPacked[queryIdx], queryPopcounts[queryIdx]
    scores = np.maximum(compute_tanimoto_similarity(qPacked, qPop, knownPacked[candidateIdx], knownPopcounts[candidateIdx], popcount8), compute_tanimoto_similarity(qPacked, qPop, knownPackedReverse[candidateIdx], knownReversePopcounts[candidateIdx], popcount8))
    validLocal = np.where(scores > minSimilarity)[0]
    if len(validLocal) == 0: return []
    topLocal = validLocal[np.argpartition(-scores[validLocal], topK - 1)[:topK]] if len(validLocal) > topK else validLocal
    topLocal = topLocal[np.argsort(-scores[topLocal])]
    return [{"queryRowId": int(queryIdx), "rank": rank, "knownRowId": int(candidateIdx[localIdx]), "rcmfpSimilarity": float(scores[localIdx]), "searchMode": searchMode, "numCandidatesSearched": int(len(candidateIdx))} for rank, localIdx in enumerate(topLocal, start=1)]

query_FP_Matrix, naturalEnzyme_fingerprint_matrix, naturalEnzyme_fingerprint_matrixReverse = query_FP_Matrix.astype(bool), naturalEnzyme_fingerprint_matrix.astype(bool), naturalEnzyme_fingerprint_matrixReverse.astype(bool)
queryPacked, knownPacked, knownPackedReverse = np.packbits(query_FP_Matrix.astype(np.uint8), axis=1), np.packbits(naturalEnzyme_fingerprint_matrix.astype(np.uint8), axis=1), np.packbits(naturalEnzyme_fingerprint_matrixReverse.astype(np.uint8), axis=1)
queryPopcounts, knownPopcounts, knownReversePopcounts = query_FP_Matrix.sum(axis=1).astype(np.int32), naturalEnzyme_fingerprint_matrix.sum(axis=1).astype(np.int32), naturalEnzyme_fingerprint_matrixReverse.sum(axis=1).astype(np.int32)
popcount8 = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)

allHitRecords = []
for queryIdx in tqdm(range(len(queryMetaDF)), desc="RCMFP enzyme retrieval"): allHitRecords.extend(retrieve_similar_reactions(queryIdx))
matchedRCMFP_DF = pd.DataFrame(allHitRecords)
print("Total hits:", len(matchedRCMFP_DF)); print(matchedRCMFP_DF["searchMode"].value_counts(dropna=False))
matchedRCMFP_DF.head()

### Assemble the full match summary

Joins query and reference metadata onto the raw hits and classifies each match by strength (Strong / Moderate / Weak / None) based on RCMFP similarity.

In [ ]:
def print_section(title):
    print("\n" + "-" * 80)
    print(title)
    print("-" * 80)

def print_saved_file(label, path):
    print(f"{label}: {path}")

def classify_precedent(sim):
    if pd.isna(sim): return "No natural precedent retrieved"
    if sim >= 0.60: return "Strong natural enzyme precedent"
    if sim >= 0.35: return "Moderate natural enzyme precedent"
    if sim > 0: return "Weak precedent; likely enzyme-engineering risk"
    return "No useful RCMFP precedent"

queryDisplayCols = [c for c in ["queryRowId", "reactionString", "ruleName", "ruleBase", "reactionType", "queryReaction", "queryMappedReaction", "candidateUniProtRaw", "numCandidateUniProt", "rcmfpMappingStatus", "rcmfpStatus", "feasibilityScore_rule1", "feasibilityLabel_rule1", "feasibilityScore_rule2", "feasibilityLabel_rule2", "feasibilityScore_rule3", "feasibilityLabel_rule3", "feasibilityScore_rule4", "feasibilityLabel_rule4"] if c in queryMetaDF.columns]
knownDisplayCols = [c for c in ["knownRowId", "knownOriginalIndex", "ruleBase", "ruleBaseSource", "rxn_idx", "mapped", "unmapped", "orig_rxn_text", "rule", "source", "quality", "natural", "organism", "protein_refs", "protein_db", "ec_num", "top_mapped_operator", "all_mapped_operators"] if c in knownMetaDF.columns]

matchedRCMFP_DF = matchedRCMFP_DF.merge(queryMetaDF[queryDisplayCols], on="queryRowId", how="left", suffixes=("", "_query")).merge(knownMetaDF[knownDisplayCols], on="knownRowId", how="left", suffixes=("", "_known")).sort_values(["queryRowId", "rank"]).reset_index(drop=True)
matchedRCMFP_DF.head()

### Merge the best natural-enzyme match per reaction to `reactionDF_wUniprotID`

Keeps the single best-scoring match per reaction (configurable via `keepMode`) and merges its columns back onto the reaction table.

In [ ]:
# User options
keyCol = "reactionString"
scoreCol = "rcmfpSimilarity"
keepMode = "best"   # "best" or "all"
bestAscending = False  # False => highest score is best, True => lowest is best

sourceDF = matchedRCMFP_DF.copy()

if keepMode == "best":
    sourceToMerge = (
        sourceDF
        .sort_values(scoreCol, ascending=bestAscending, na_position="last")
        .drop_duplicates(subset=[keyCol], keep="first")
    )
elif keepMode == "all":
    sourceToMerge = sourceDF
else:
    raise ValueError("keepMode must be 'best' or 'all'")

# only add columns that don't already exist in target
newCols = [c for c in sourceToMerge.columns if c not in reactionDF_wUniprotID.columns and c != keyCol]

reactionDF_wUniprotID = reactionDF_wUniprotID.merge(
    sourceToMerge[[keyCol] + newCols],
    on=keyCol,
    how="inner"   # keep only rows present in source
)

print(f"Mode: {keepMode}")
print(f"Rows after merge: {len(reactionDF_wUniprotID):,}")
print(f"Added columns: {len(newCols)}")
reactionDF_wUniprotID

### Find best `UniProt ID` per DORANet reaction rule

Extracts UniProt accessions from the matched references' protein references, then ranks them per rule by number of supporting hits and best RCMFP similarity.

In [ ]:
uniprotPattern = re.compile(
    r"([OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9](?:[A-Z0-9]{3}[0-9]){1,2})"
)

def extractUniProtID(x):
    if pd.isna(x):
        return []
    ids = uniprotPattern.findall(str(x))
    seen, out = set(), []
    for uid in ids:
        if uid not in seen:
            out.append(uid)
            seen.add(uid)
    return out


tmp = matchedRCMFP_DF.copy()

tmp["knownProteinRefList"] = tmp["protein_refs"].apply(extractUniProtID)

tmp = tmp[
    tmp["knownProteinRefList"].apply(len) > 0
].copy()

tmp = tmp.explode("knownProteinRefList").rename(
    columns={"knownProteinRefList": "rcmfpSupportedUniProt"}
)

tmp["rcmfpSimilarity"] = pd.to_numeric(
    tmp["rcmfpSimilarity"],
    errors="coerce"
)

tmp["rank"] = pd.to_numeric(
    tmp["rank"],
    errors="coerce"
)

ruleRcmfpUniProtScoreDF = (
    tmp
    .groupby(["ruleName", "ruleBase", "rcmfpSupportedUniProt"])
    .agg(
        numSupportingHits=("knownRowId", "nunique"),
        bestRCMFPSimilarity=("rcmfpSimilarity", "max"),
        meanRCMFPSimilarity=("rcmfpSimilarity", "mean"),
        bestRank=("rank", "min"),
        exampleKnownRowId=("knownRowId", "first"),
        exampleKnownReaction=("orig_rxn_text", "first"),
        exampleKnownOrganism=("organism", "first"),
        exampleKnownEcNumber=("ec_num", "first"),
        exampleProteinRefs=("protein_refs", "first"),
    )
    .reset_index()
)

ruleRcmfpUniProtScoreDF["rcmfpProteinRefScore"] = (
    ruleRcmfpUniProtScoreDF["bestRCMFPSimilarity"] * 100
    + np.log1p(ruleRcmfpUniProtScoreDF["numSupportingHits"]) * 10
    - ruleRcmfpUniProtScoreDF["bestRank"] * 0.1
)

ruleRcmfpUniProtScoreDF = ruleRcmfpUniProtScoreDF.sort_values(
    [
        "ruleName",
        "rcmfpProteinRefScore",
        "bestRCMFPSimilarity",
        "numSupportingHits",
        "bestRank",
    ],
    ascending=[True, False, False, False, True]
)

bestRCMFPUniProtByRuleDF = (
    ruleRcmfpUniProtScoreDF
    .groupby("ruleName")
    .head(1)
    .reset_index(drop=True)
)

bestRCMFPUniProtByRuleDF[
    [
        "ruleName",
        "rcmfpSupportedUniProt",
        "rcmfpProteinRefScore",
        "bestRCMFPSimilarity",
        "meanRCMFPSimilarity",
        "numSupportingHits",
        "bestRank",
        "exampleKnownOrganism",
        "exampleKnownEcNumber",
        "exampleKnownReaction",
    ]
].head()

### Merge best `RCMFP-supported UniProt ID` with `reactionDF_wUniprotID`

Attaches the top UniProt ID (and its supporting scores) to every reaction sharing that rule.

In [ ]:
# Columns to keep + rename map
rename_map = {
    "rcmfpSupportedUniProt": "bestRCMFP_UniProtID",
    "rcmfpProteinRefScore": "bestRCMFP_UniProtScore",
    "bestRCMFPSimilarity": "bestRCMFP_SimilarityScore",
    "meanRCMFPSimilarity": "meanRCMFP_Similarity_byRule",
    "numSupportingHits": "numRCMFP_SupportingHits_byRule",
    "bestRank": "bestRCMFPRank_byRule",
    "exampleKnownRowId": "bestRCMFP_RowId",
    "exampleKnownReaction": "bestRCMFP_Reaction",
    "exampleKnownOrganism": "bestRCMFP_Organism",
    "exampleKnownEcNumber": "bestRCMFP_ECnumber",
    "exampleProteinRefs": "bestRCMFP_ProteinRefs",
}

bestRCMFPForMergeDF = (
    bestRCMFPUniProtByRuleDF[["ruleName", *rename_map]]
    .rename(columns=rename_map)
)

reactionDF_wBestRCMFP_Similarity = (
    reactionDF_wUniprotID
    .merge(bestRCMFPForMergeDF, on="ruleName", how="left")
    .assign(
        hasbestRCMFP_UniProtID=lambda df: (
            df["bestRCMFP_UniProtID"].fillna("").astype(str).str.len().gt(0)
        )
    )
)

print("reactionDF_wBestRCMFP_Similarity created")
print(f"Rows in reactionDF_wUniprotID              : {len(reactionDF_wUniprotID):,}")
print(f"Rows in reactionDF_wBestRCMFP_Similarity   : {len(reactionDF_wBestRCMFP_Similarity):,}")
print(f"Rows with best RCMFP UniProt               : {reactionDF_wBestRCMFP_Similarity['hasbestRCMFP_UniProtID'].sum():,}")
print(f"Unique best RCMFP UniProt IDs              : {reactionDF_wBestRCMFP_Similarity['bestRCMFP_UniProtID'].nunique():,}")

reactionDF_wBestRCMFP_Similarity.head()

### Drop intermediate columns

Removes bookkeeping columns to keep the reaction table readable for the next stage.

In [ ]:
reactionDF_wBestRCMFP_Similarity = reactionDF_wBestRCMFP_Similarity.drop(
    columns=[
        'reactantSetCanonical', 'productSetCanonical',
        'rxn_str', 'feasibilityScore_rule1', 'feasibilityLabel_rule1',
        'feasibilityLabel_rule2', 'feasibilityScore_rule3', 'feasibilityLabel_rule3',
        'feasibilityScore_rule4', 'feasibilityLabel_rule4', 'ruleReactants',
        'ruleProducts', 'ruleSMARTS', 'hasRuleLookup', 'queryRowId', 'rank',
        'knownRowId', 'rcmfpSimilarity', 'searchMode', 'numCandidatesSearched',
        'ruleBase', 'queryReaction', 'queryMappedReaction','rcmfpMappingStatus', 'rcmfpStatus', 'knownOriginalIndex',
        'ruleBase_known', 'ruleBaseSource', 'rxn_idx', 'mapped', 'unmapped','orig_rxn_text', 'rule', 'source', 'quality', 'natural', 
        'organism','protein_refs', 'protein_db', 'ec_num', 'top_mapped_operator',
        'all_mapped_operators','bestRCMFP_UniProtScore', 'meanRCMFP_Similarity_byRule','numRCMFP_SupportingHits_byRule',      
        'bestRCMFPRank_byRule','bestRCMFP_RowId', 'bestRCMFP_Reaction', 'bestRCMFP_ProteinRefs', 'hasbestRCMFP_UniProtID'               
    ],
    errors='ignore'  
)

reactionDF_wBestRCMFP_Similarity

## 5. Query UniProt for EC numbers, protein names, organisms and sequences

```text
DORAnet rule ID → UniProt accession → EC number/protein sequence
```

### Summarize rules with a prioritized UniProt ID

Prefers the RCMFP-supported UniProt ID, falling back to the rule's candidate ID, then summarizes usage per rule.

In [ ]:
tmp = reactionDF_wBestRCMFP_Similarity.copy()

tmp["prioritizedUniProt"] = (
    tmp["bestRCMFP_UniProtID"].fillna("").astype(str).str.strip()
)
fallback_mask = tmp["prioritizedUniProt"].eq("")
tmp.loc[fallback_mask, "prioritizedUniProt"] = (
    tmp.loc[fallback_mask, "candidateUniProtRaw"].fillna("").astype(str).str.strip()
)

usedRuleSummaryDF = (
    tmp
    .groupby(["ruleName", "reactionType", "prioritizedUniProt"], dropna=False)
    .agg(
        numReactions=("reactionString", "count"),
        exampleReaction=("reactionString", "first"),
    )
    .reset_index()
    .sort_values("numReactions", ascending=False)
)

usedRuleSummaryDF

### Fetch UniProt metadata

Queries the UniProt REST API in chunks for EC numbers, protein names, organisms, lengths, and sequences, then merges the results back onto the reaction table.

In [ ]:
def chunkList(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]


def cleanUniProtAccession(x):
    if pd.isna(x):
        return None

    x = str(x).strip()
    x = x.replace("UniProtKB:", "").replace("UniProt:", "").replace("uniprot:", "")
    x = x.replace("sp|", "").replace("tr|", "")

    # Handles strings like sp|P77791|NAME
    if "|" in x:
        parts = [p.strip() for p in x.split("|") if p.strip()]
        for p in parts:
            if isValidUniProtAccession(p):
                return p

    x = x.split()[0].split("-")[0].strip()
    return x if x else None


uniprotAccessionPattern = re.compile(
    r"^([OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9]([A-Z0-9]{3}[0-9]){1,2})$"
)


def isValidUniProtAccession(x):
    return False if pd.isna(x) else bool(
        uniprotAccessionPattern.match(str(x).strip())
    )


def fetchUniProtMetadata(accessionList, chunkSize=20, sleepSeconds=0.5):
    fields = [
        "accession",
        "reviewed",
        "id",
        "protein_name",
        "gene_names",
        "organism_name",
        "organism_id",
        "ec",
        "length",
        "sequence",
    ]

    allDf = []
    errors = 0

    for chunk in tqdm(list(chunkList(accessionList, chunkSize)), desc="Querying UniProt"):
        q = " OR ".join([f"(accession_id:{a})" for a in chunk])

        try:
            r = requests.get(
                "https://rest.uniprot.org/uniprotkb/search",
                params={
                    "query": q,
                    "format": "tsv",
                    "fields": ",".join(fields),
                    "size": chunkSize,
                },
                timeout=120,
            )

            if r.status_code != 200:
                errors += 1
            else:
                df = pd.read_csv(StringIO(r.text), sep="\t")

                if len(df) == 0:
                    errors += 1
                else:
                    allDf.append(df)

        except Exception:
            errors += 1

        time.sleep(sleepSeconds)

    out = (
        pd.concat(allDf, ignore_index=True).drop_duplicates()
        if allDf
        else pd.DataFrame()
    )

    return out, errors


# -----------------------------------------------
# Filter best RCMFP UniProt IDs
# -----------------------------------------------
reactionDF_wBestRCMFP_Similarity = reactionDF_wBestRCMFP_Similarity.copy()

reactionDF_wBestRCMFP_Similarity["bestRCMFP_UniProtID_clean"] = (
    reactionDF_wBestRCMFP_Similarity["bestRCMFP_UniProtID"]
    .apply(cleanUniProtAccession)
)

reactionDF_wBestRCMFP_Similarity["isValidBestRCMFP_UniProtID"] = (
    reactionDF_wBestRCMFP_Similarity["bestRCMFP_UniProtID_clean"]
    .apply(isValidUniProtAccession)
)

uniqueAccessionList = sorted(
    reactionDF_wBestRCMFP_Similarity.loc[
        reactionDF_wBestRCMFP_Similarity["isValidBestRCMFP_UniProtID"],
        "bestRCMFP_UniProtID_clean",
    ]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

print(f"Unique valid best RCMFP UniProt IDs: {len(uniqueAccessionList):,}")

if not uniqueAccessionList:
    raise RuntimeError("No valid best RCMFP UniProt IDs found.")


# -----------------------------------------------
# Query UniProt
# -----------------------------------------------
uniprotMetadataRawDF, failedChunks = fetchUniProtMetadata(
    uniqueAccessionList,
    chunkSize=20,
    sleepSeconds=0.5,
)

totalChunks = int(np.ceil(len(uniqueAccessionList) / 20))
successChunks = totalChunks - failedChunks

print(f"Successful query chunks: {successChunks}")
print(f"Failed query chunks    : {failedChunks}")


# -----------------------------------------------
# Standardize UniProt columns
# -----------------------------------------------
renameMap = {
    "Entry": "uniprotAccession",
    "Reviewed": "uniprotReviewed",
    "Entry Name": "entryName",
    "Protein names": "proteinName",
    "Gene Names": "geneNames",
    "Organism": "organism",
    "Organism (ID)": "organismTaxId",
    "EC number": "ecNumber",
    "Length": "proteinLengthAa",
    "Sequence": "proteinSequence",
}

uniprotMetadataDF = uniprotMetadataRawDF.rename(columns=renameMap)

requiredCols = [
    "uniprotAccession",
    "uniprotReviewed",
    "entryName",
    "proteinName",
    "geneNames",
    "organism",
    "organismTaxId",
    "ecNumber",
    "proteinLengthAa",
    "proteinSequence",
]

for c in requiredCols:
    if c not in uniprotMetadataDF.columns:
        uniprotMetadataDF[c] = np.nan

uniprotMetadataDF = uniprotMetadataDF[requiredCols].copy()


# -----------------------------------------------
# Merge UniProt metadata back to reactionDF_wBestRCMFP_Similarity
# -----------------------------------------------
reactionDF_wBestRCMFP_Similarity = reactionDF_wBestRCMFP_Similarity.rename(
    columns={"bestRCMFP_UniProtID_clean": "uniprotAccession"}
)

reactionDF_wBestRCMFP_wUniProtMetadata = reactionDF_wBestRCMFP_Similarity.merge(
    uniprotMetadataDF,
    on="uniprotAccession",
    how="left",
    suffixes=("", "_uniprot"),
)

reactionDF_wBestRCMFP_wUniProtMetadata["hasUniProtMetadata"] = (
    reactionDF_wBestRCMFP_wUniProtMetadata["proteinName"]
    .fillna("")
    .astype(str)
    .str.len() > 0
)

reactionDF_wBestRCMFP_wUniProtMetadata["hasProteinSequence"] = (
    reactionDF_wBestRCMFP_wUniProtMetadata["proteinSequence"]
    .fillna("")
    .astype(str)
    .str.len() > 0
)

print("reactionDF_wBestRCMFP_wUniProtMetadata created")
print(f"Rows total             : {len(reactionDF_wBestRCMFP_wUniProtMetadata):,}")
print(f"Rows with metadata     : {reactionDF_wBestRCMFP_wUniProtMetadata['hasUniProtMetadata'].sum():,}")
print(f"Rows with protein seq  : {reactionDF_wBestRCMFP_wUniProtMetadata['hasProteinSequence'].sum():,}")

reactionDF_wBestRCMFP_wUniProtMetadata.head()

## 6. Fetch `DNA sequences` from `GenBank`

For each UniProt accession, follows its RefSeq/EMBL/GenBank cross-references and pulls the coding DNA sequence from NCBI. Requires a valid `NCBI_EMAIL` (set in CONFIG). Set `maxUniProtToFetch` to a small number for a test run.

In [ ]:
NCBI_TOOL = "DORAnet_GenBank_CDS_Fetch"

ncbiSleepSeconds = 0.40 if NCBI_API_KEY is None else 0.12
uniprotSleepSeconds = 0.15

# For testing, set 20 or 50. For full run, keep None.
maxUniProtToFetch = None



# UniProt column setup
if "reactionDF_wBestRCMFP_wUniProtMetadata" not in globals():
    raise RuntimeError("reactionDF_wBestRCMFP_wUniProtMetadata is not defined.")

reactionDF_wBestRCMFP_wUniProtMetadata = reactionDF_wBestRCMFP_wUniProtMetadata.copy()

# Prefer existing uniprotAccession column.
# Otherwise use bestRCMFP_UniProtID.
if "uniprotAccession" not in reactionDF_wBestRCMFP_wUniProtMetadata.columns:
    if "bestRCMFP_UniProtID" in reactionDF_wBestRCMFP_wUniProtMetadata.columns:
        reactionDF_wBestRCMFP_wUniProtMetadata["uniprotAccession"] = (
            reactionDF_wBestRCMFP_wUniProtMetadata["bestRCMFP_UniProtID"]
        )
    else:
        raise ValueError("Need either 'uniprotAccession' or 'bestRCMFP_UniProtID' column.")


# Helper functions
def dedupeKeepOrder(valueList):
    seen = set()
    out = []

    for value in valueList:
        if value is None:
            continue

        value = str(value).strip()

        if value and value not in seen:
            out.append(value)
            seen.add(value)

    return out


def cleanUniProtAccession(x):
    if pd.isna(x):
        return None

    x = str(x).strip()
    x = (
        x.replace("UniProtKB:", "")
        .replace("UniProt:", "")
        .replace("uniprot:", "")
        .replace("sp|", "")
        .replace("tr|", "")
    )

    if "|" in x:
        parts = [p.strip() for p in x.split("|") if p.strip()]
        for p in parts:
            if isValidUniProtAccession(p):
                return p

    x = x.split()[0].split("-")[0].strip()

    return x if x else None


uniprotAccessionPattern = re.compile(
    r"^([OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9]([A-Z0-9]{3}[0-9]){1,2})$"
)


def isValidUniProtAccession(x):
    return False if pd.isna(x) else bool(
        uniprotAccessionPattern.match(str(x).strip())
    )


def parseFastaRecords(fastaText):
    records = []
    header = None
    seqLines = []

    for line in str(fastaText).splitlines():
        line = line.strip()

        if not line:
            continue

        if line.startswith(">"):
            if header is not None:
                records.append({
                    "header": header,
                    "sequence": "".join(seqLines),
                })

            header = line[1:]
            seqLines = []

        else:
            seqLines.append(line)

    if header is not None:
        records.append({
            "header": header,
            "sequence": "".join(seqLines),
        })

    return records


def cleanDnaSequence(seq):
    seq = str(seq).upper()
    seq = re.sub(r"[^ACGTN]", "", seq)
    return seq


def fetchUniProtCrossRefs(uniprotAccession, sleepSeconds=0.15):
    url = f"https://rest.uniprot.org/uniprotkb/{uniprotAccession}.json"

    result = {
        "uniprotAccession": uniprotAccession,
        "refseqProteinIds": [],
        "emblProteinIds": [],
        "nucleotideIds": [],
        "rawCrossRefSummary": "",
        "uniprotCrossRefStatus": "not_queried",
        "uniprotCrossRefError": "",
    }

    try:
        response = requests.get(url, timeout=60)

        if response.status_code != 200:
            result["uniprotCrossRefStatus"] = "failed"
            result["uniprotCrossRefError"] = f"{response.status_code}: {response.text[:300]}"
            time.sleep(sleepSeconds)
            return result

        entryJson = response.json()
        crossRefs = entryJson.get("uniProtKBCrossReferences", [])

        refseqProteinIds = []
        emblProteinIds = []
        nucleotideIds = []
        rawSummaryParts = []

        for xref in crossRefs:
            databaseName = str(xref.get("database", "")).strip()
            xrefId = str(xref.get("id", "")).strip()
            properties = xref.get("properties", [])

            propertyDict = {}

            for prop in properties:
                key = str(prop.get("key", "")).strip()
                value = str(prop.get("value", "")).strip()

                if key and value:
                    propertyDict.setdefault(key, []).append(value)

            if databaseName in ["RefSeq", "EMBL", "GenBank", "DDBJ"]:
                rawSummaryParts.append(f"{databaseName}:{xrefId}")

            if databaseName == "RefSeq" and xrefId:
                refseqProteinIds.append(xrefId)

            if databaseName in ["EMBL", "GenBank", "DDBJ"]:
                if xrefId:
                    nucleotideIds.append(xrefId)

                for key, values in propertyDict.items():
                    normalizedKey = (
                        key.lower()
                        .replace(" ", "")
                        .replace("_", "")
                        .replace("-", "")
                    )

                    if "proteinid" in normalizedKey or normalizedKey == "protein":
                        emblProteinIds.extend(values)

                    if "nucleotidesequenceid" in normalizedKey or "nucleotide" in normalizedKey:
                        nucleotideIds.extend(values)

        result["refseqProteinIds"] = dedupeKeepOrder(refseqProteinIds)
        result["emblProteinIds"] = dedupeKeepOrder(emblProteinIds)
        result["nucleotideIds"] = dedupeKeepOrder(nucleotideIds)
        result["rawCrossRefSummary"] = ";".join(rawSummaryParts)
        result["uniprotCrossRefStatus"] = "success"

    except Exception as exc:
        result["uniprotCrossRefStatus"] = "failed"
        result["uniprotCrossRefError"] = str(exc)

    time.sleep(sleepSeconds)

    return result


def fetchNcbiCdsFromProteinAccession(proteinAccession, sleepSeconds=0.40):
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"

    params = {
        "db": "protein",
        "id": proteinAccession,
        "rettype": "fasta_cds_na",
        "retmode": "text",
        "tool": NCBI_TOOL,
        "email": NCBI_EMAIL,
    }

    if NCBI_API_KEY is not None:
        params["api_key"] = NCBI_API_KEY

    result = {
        "proteinAccessionQueried": proteinAccession,
        "ncbiFetchStatus": "not_queried",
        "ncbiFetchError": "",
        "genbankCdsFastaHeader": "",
        "genbankCdsSequence": "",
        "genbankCdsLengthBp": np.nan,
        "numFastaRecordsReturned": 0,
    }

    try:
        response = requests.get(url, params=params, timeout=120)

        if response.status_code != 200:
            result["ncbiFetchStatus"] = "failed"
            result["ncbiFetchError"] = f"{response.status_code}: {response.text[:300]}"
            time.sleep(sleepSeconds)
            return result

        fastaRecords = parseFastaRecords(response.text.strip())

        if len(fastaRecords) == 0:
            result["ncbiFetchStatus"] = "no_cds_returned"
            result["ncbiFetchError"] = response.text[:300]
            time.sleep(sleepSeconds)
            return result

        firstRecord = fastaRecords[0]
        cleanSeq = cleanDnaSequence(firstRecord["sequence"])

        if len(cleanSeq) == 0:
            result["ncbiFetchStatus"] = "empty_cds_sequence"
            result["ncbiFetchError"] = "No valid DNA sequence parsed."
            time.sleep(sleepSeconds)
            return result

        result["ncbiFetchStatus"] = "success"
        result["genbankCdsFastaHeader"] = firstRecord["header"]
        result["genbankCdsSequence"] = cleanSeq
        result["genbankCdsLengthBp"] = len(cleanSeq)
        result["numFastaRecordsReturned"] = len(fastaRecords)

    except Exception as exc:
        result["ncbiFetchStatus"] = "failed"
        result["ncbiFetchError"] = str(exc)

    time.sleep(sleepSeconds)

    return result


def fetchBestGenBankCdsForUniProt(uniprotAccession):
    crossRefResult = fetchUniProtCrossRefs(
        uniprotAccession,
        sleepSeconds=uniprotSleepSeconds,
    )

    candidateProteinIds = dedupeKeepOrder(
        crossRefResult["refseqProteinIds"] + crossRefResult["emblProteinIds"]
    )

    output = {
        "uniprotAccession": uniprotAccession,
        "genbankProteinIdsFromUniProt": ";".join(candidateProteinIds),
        "genbankNucleotideIdsFromUniProt": ";".join(crossRefResult["nucleotideIds"]),
        "rawCrossRefSummary": crossRefResult["rawCrossRefSummary"],
        "uniprotCrossRefStatus": crossRefResult["uniprotCrossRefStatus"],
        "uniprotCrossRefError": crossRefResult["uniprotCrossRefError"],
        "genbankProteinAccessionUsed": "",
        "genbankCdsFastaHeader": "",
        "genbankCdsSequence": "",
        "genbankCdsLengthBp": np.nan,
        "numFastaRecordsReturned": 0,
        "genbankCdsFetchStatus": "not_attempted",
        "genbankCdsFetchError": "",
    }

    if len(candidateProteinIds) == 0:
        output["genbankCdsFetchStatus"] = "no_protein_crossref"
        output["genbankCdsFetchError"] = (
            "No RefSeq/EMBL/GenBank protein cross-reference found in UniProt JSON."
        )
        return output

    fetchErrors = []

    for proteinAccession in candidateProteinIds:
        cdsResult = fetchNcbiCdsFromProteinAccession(
            proteinAccession,
            sleepSeconds=ncbiSleepSeconds,
        )

        if cdsResult["ncbiFetchStatus"] == "success":
            output["genbankProteinAccessionUsed"] = proteinAccession
            output["genbankCdsFastaHeader"] = cdsResult["genbankCdsFastaHeader"]
            output["genbankCdsSequence"] = cdsResult["genbankCdsSequence"]
            output["genbankCdsLengthBp"] = cdsResult["genbankCdsLengthBp"]
            output["numFastaRecordsReturned"] = cdsResult["numFastaRecordsReturned"]
            output["genbankCdsFetchStatus"] = "success"
            output["genbankCdsFetchError"] = ""
            return output

        fetchErrors.append(
            f"{proteinAccession}: {cdsResult['ncbiFetchStatus']} | {cdsResult['ncbiFetchError']}"
        )

    output["genbankCdsFetchStatus"] = "failed_all_protein_crossrefs"
    output["genbankCdsFetchError"] = " || ".join(fetchErrors[:5])

    return output


# Choose UniProt IDs from reactionDF_wBestRCMFP_wUniProtMetadata
reactionDF_wBestRCMFP_wUniProtMetadata["uniprotAccession"] = (
    reactionDF_wBestRCMFP_wUniProtMetadata["uniprotAccession"]
    .apply(cleanUniProtAccession)
)

reactionDF_wBestRCMFP_wUniProtMetadata["isValidUniProtAccession"] = (
    reactionDF_wBestRCMFP_wUniProtMetadata["uniprotAccession"]
    .apply(isValidUniProtAccession)
)

uniqueUniProtForGenBankList = sorted(
    reactionDF_wBestRCMFP_wUniProtMetadata.loc[
        reactionDF_wBestRCMFP_wUniProtMetadata["isValidUniProtAccession"],
        "uniprotAccession",
    ]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

if maxUniProtToFetch is not None:
    uniqueUniProtForGenBankList = uniqueUniProtForGenBankList[:maxUniProtToFetch]

print(f"Unique UniProt accessions selected for GenBank CDS fetch: {len(uniqueUniProtForGenBankList):,}")

if len(uniqueUniProtForGenBankList) == 0:
    raise RuntimeError("No valid UniProt accessions found for GenBank CDS fetch.")


# Fetch GenBank CDS sequences
genbankLookupRecords = []

for uniprotAccession in tqdm(
    uniqueUniProtForGenBankList,
    desc="Fetching GenBank CDS via UniProt crossrefs",
):
    record = fetchBestGenBankCdsForUniProt(uniprotAccession)
    genbankLookupRecords.append(record)

genbankLookupDF = pd.DataFrame(genbankLookupRecords)

print("\nGenBank CDS fetch status:")
print(genbankLookupDF["genbankCdsFetchStatus"].value_counts(dropna=False))


# Merge GenBank CDS back to reaction dataframe
reactionDF_wBestRCMFP_wDNAseq = reactionDF_wBestRCMFP_wUniProtMetadata.merge(
    genbankLookupDF,
    on="uniprotAccession",
    how="left",
)

reactionDF_wBestRCMFP_wDNAseq["hasGenBankCdsSequence"] = (
    reactionDF_wBestRCMFP_wDNAseq["genbankCdsSequence"]
    .fillna("")
    .astype(str)
    .str.len() > 0
)

print("\nreactionDF_wBestRCMFP_wDNAseq created")
print(f"Rows total              : {len(reactionDF_wBestRCMFP_wDNAseq):,}")
print(f"Rows with GenBank CDS   : {reactionDF_wBestRCMFP_wDNAseq['hasGenBankCdsSequence'].sum():,}")

reactionDF_wBestRCMFP_wDNAseq.head()

### Drop reactions without a DNA sequence

Keeps only reactions for which a GenBank CDS was retrieved.

In [ ]:
before_rows = len(reactionDF_wBestRCMFP_wDNAseq)

reactionDF_wBestRCMFP_wDNAseq = reactionDF_wBestRCMFP_wDNAseq[reactionDF_wBestRCMFP_wDNAseq["hasGenBankCdsSequence"] != False].copy()
after_rows = len(reactionDF_wBestRCMFP_wDNAseq)
dropped_rows = before_rows - after_rows

print(f"Total reaction : {before_rows:,}")
print(f"Reaction with GenBank CDS  : {after_rows:,}")
print(f"Reaction dropped : {dropped_rows:,}")

## 7. Codon-optimize selected enzymes for `target_species` using `DNA Chisel`

### Inspect source organisms

Shows the distribution of organisms behind the retrieved sequences — useful context before codon optimization.

In [ ]:
organismCountsDF = (
    reactionDF_wBestRCMFP_wDNAseq["organism"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
    .value_counts(dropna=False)
    .rename_axis("organism")
    .reset_index(name="count")
)

totalRows = len(reactionDF_wBestRCMFP_wDNAseq)
organismCountsDF["percentage"] = (organismCountsDF["count"] / totalRows * 100).round(2)

print(f"Unique organisms: {organismCountsDF['organism'].nunique()}\n")
organismCountsDF

### Codon optimization settings

In [ ]:
# targetSpecies, minGc, maxGc, gcWindow, and stopCodon are set in the CONFIG cell.
# Restriction sites to avoid in the optimized CDS (edit for your assembly standard):
forbiddenPatternList = [
    "BsaI_site",
    "BsmBI_site",
    "EcoRI_site",
    "XbaI_site",
    "SpeI_site",
    "PstI_site",
    "NotI_site",
]

stopCodons = {"TAA", "TAG", "TGA"}

### Codon optimize with DNA Chisel

Cleans each CDS, screens out sequences that can't be optimized (internal stops, non-triplet length, ambiguous bases), then runs DNA Chisel to enforce translation, GC content, and restriction-site avoidance while maximizing codon-adaptation index for the target host. Optimization runs once per **unique** CDS.

In [ ]:
# Helper functions
def cleanDna(seq):
    if pd.isna(seq):
        return ""
    return re.sub(r"[^ACGTN]", "", str(seq).upper())


def removeTerminalStop(cds):
    cds = cleanDna(cds)
    if len(cds) >= 3 and len(cds) % 3 == 0 and cds[-3:] in stopCodons:
        return cds[:-3]
    return cds


def hasInternalStop(cds):
    cds = cleanDna(cds)
    if len(cds) % 3 != 0:
        return True
    codons = [cds[i:i+3] for i in range(0, len(cds), 3)]
    return any(codon in stopCodons for codon in codons)


def classifyCdsForOptimization(cds):
    cds = cleanDna(cds)
    cdsNoStop = removeTerminalStop(cds)

    if len(cds) == 0:
        return "no_cds"
    if len(cdsNoStop) == 0:
        return "empty_after_stop_removal"
    if len(cdsNoStop) % 3 != 0:
        return "length_not_multiple_of_3"
    if "N" in cdsNoStop:
        return "contains_N"
    if hasInternalStop(cdsNoStop):
        return "contains_internal_stop"

    return "ready_for_optimization"


def safeText(x):
    if pd.isna(x):
        return ""
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(x))[:80]


def wrapFasta(seq, width=80):
    seq = str(seq)
    return "\n".join(seq[i:i+width] for i in range(0, len(seq), width))


def optimizeCdsForEcoli(initialCds):
    sequenceLength = len(initialCds)
    geneLocation = (0, sequenceLength)

    constraints = [
        EnforceTranslation(location=geneLocation),
        EnforceGCContent(mini=minGc, maxi=maxGc, window=gcWindow),
    ]

    for patternName in forbiddenPatternList:
        constraints.append(AvoidPattern(patternName))

    objectives = [
        MaximizeCAI(species=targetSpecies, location=geneLocation)
    ]

    optimizationProblem = DnaOptimizationProblem(
        sequence=initialCds,
        constraints=constraints,
        objectives=objectives,
    )

    optimizationProblem.resolve_constraints()
    optimizationProblem.optimize()

    return optimizationProblem.sequence, optimizationProblem


# Prepare input from reactionDF_wBestRCMFP_wDNAseq
reactionDF_wBestRCMFP_wDNAseq_forOpt = reactionDF_wBestRCMFP_wDNAseq.copy()

reactionDF_wBestRCMFP_wDNAseq_forOpt["genbankCdsSequence"] = (
    reactionDF_wBestRCMFP_wDNAseq_forOpt["genbankCdsSequence"]
    .apply(cleanDna)
)

reactionDF_wBestRCMFP_wDNAseq_forOpt["genbankCdsSequenceForOptimization"] = (
    reactionDF_wBestRCMFP_wDNAseq_forOpt["genbankCdsSequence"]
    .apply(removeTerminalStop)
)

reactionDF_wBestRCMFP_wDNAseq_forOpt["codonOptimizationInputStatus"] = (
    reactionDF_wBestRCMFP_wDNAseq_forOpt["genbankCdsSequence"]
    .apply(classifyCdsForOptimization)
)

reactionDF_wBestRCMFP_wDNAseq_forOpt["isReadyForCodonOptimization"] = (
    reactionDF_wBestRCMFP_wDNAseq_forOpt["codonOptimizationInputStatus"]
    == "ready_for_optimization"
)

reactionDF_wBestRCMFP_wDNAseq_forOpt["optimizationInputKey"] = (
    reactionDF_wBestRCMFP_wDNAseq_forOpt["uniprotAccession"].fillna("").astype(str)
    + "|"
    + reactionDF_wBestRCMFP_wDNAseq_forOpt["genbankProteinAccessionUsed"].fillna("").astype(str)
    + "|"
    + reactionDF_wBestRCMFP_wDNAseq_forOpt["genbankCdsSequenceForOptimization"].fillna("").astype(str)
).apply(lambda x: hashlib.sha1(x.encode("utf-8")).hexdigest())

print("Input status:")
print(reactionDF_wBestRCMFP_wDNAseq_forOpt["codonOptimizationInputStatus"].value_counts(dropna=False))


# Optimize unique GenBank CDS records only
geneInputDF = (
    reactionDF_wBestRCMFP_wDNAseq_forOpt[
        reactionDF_wBestRCMFP_wDNAseq_forOpt["isReadyForCodonOptimization"]
    ]
    .drop_duplicates("optimizationInputKey")
    .reset_index(drop=True)
)

optimizedGeneRecords = []

for idx, row in tqdm(
    geneInputDF.iterrows(),
    total=len(geneInputDF),
    desc="Optimizing RCMFP-supported GenBank CDS for E. coli"
):
    optimizationInputKey = row["optimizationInputKey"]
    initialCds = row["genbankCdsSequenceForOptimization"]

    try:
        optimizedCdsNoStop, optimizationProblem = optimizeCdsForEcoli(initialCds)
        optimizedCds = optimizedCdsNoStop + stopCodon

        optimizationStatus = "success"
        optimizationError = ""
        constraintsSummary = optimizationProblem.constraints_text_summary()
        objectivesSummary = optimizationProblem.objectives_text_summary()

    except Exception as exc:
        optimizedCds = ""
        optimizationStatus = "failed"
        optimizationError = str(exc)
        constraintsSummary = ""
        objectivesSummary = ""

    optimizedGeneRecords.append({
        "optimizationInputKey": optimizationInputKey,
        "optimizedGeneId": f"ecoli_rcmfp_opt_gene_{idx:06d}",
        "optimizedCds": optimizedCds,
        "optimizedCdsLengthBp": len(optimizedCds),
        "expressionHost": "Escherichia coli",
        "targetSpecies": targetSpecies,
        "minGc": minGc,
        "maxGc": maxGc,
        "gcWindow": gcWindow,
        "stopCodonUsed": stopCodon,
        "codonOptimizationStatus": optimizationStatus,
        "codonOptimizationError": optimizationError,
        "constraintsSummary": constraintsSummary,
        "objectivesSummary": objectivesSummary,
    })

optimizedGeneDF = pd.DataFrame(optimizedGeneRecords)

print("\nOptimization status:")
print(optimizedGeneDF["codonOptimizationStatus"].value_counts(dropna=False))


# Merge optimized CDS back to reaction dataframe
reactionDF_wOptimizedDNAseq = reactionDF_wBestRCMFP_wDNAseq_forOpt.merge(
    optimizedGeneDF,
    on="optimizationInputKey",
    how="left"
)

reactionDF_wOptimizedDNAseq["hasOptimizedCds"] = (
    reactionDF_wOptimizedDNAseq["optimizedCds"]
    .fillna("")
    .astype(str)
    .str.len() > 0
)

print("\nFinal reactionDF_wOptimizedDNAseq:")
print(f"Rows total           : {len(reactionDF_wOptimizedDNAseq):,}")
print(f"Rows optimized       : {reactionDF_wOptimizedDNAseq['hasOptimizedCds'].sum():,}")
print(f"Unique optimized CDS : {reactionDF_wOptimizedDNAseq.loc[reactionDF_wOptimizedDNAseq['hasOptimizedCds'], 'optimizationInputKey'].nunique():,}")
reactionDF_wOptimizedDNAseq.head()

## 8. Export `DNA design` files for `Teselagen`

### Export optimized sequences to FASTA

Writes one FASTA record per unique optimized gene, with rule, UniProt, GenBank, EC, and host annotations in the header. The file name reflects the target host automatically.

In [ ]:
optimizedDNAFilePath = os.path.join(
    DNADesignResultsDir, f"reactionDF_wBestRCMFP_optimized_{targetSpecies}.fasta"
)

fastaDF = (reactionDF_wOptimizedDNAseq[reactionDF_wOptimizedDNAseq["hasOptimizedCds"]].drop_duplicates("optimizationInputKey").copy())

with open(optimizedDNAFilePath, "w", encoding="utf-8") as f:
    for _, row in fastaDF.iterrows():
        fastaHeader = (
            f">{row['optimizedGeneId']}|"
            f"rule={safeText(row.get('ruleName', ''))}|"
            f"UniProt={safeText(row.get('uniprotAccession', ''))}|"
            f"GenBankProtein={safeText(row.get('genbankProteinAccessionUsed', ''))}|"
            f"EC={safeText(row.get('ecNumber', ''))}|"
            f"host={safeText(targetSpecies)}"
        )

        f.write(fastaHeader + "\n")
        f.write(wrapFasta(row["optimizedCds"]) + "\n\n")

print(f"\nSaved FASTA at: {optimizedDNAFilePath}")

### Recount complete `starter → target` pathways (post-optimization)

Recomputes route-level pathway counts using only reactions that survived to the optimized-DNA stage.

In [ ]:
df = reactionDF_wOptimizedDNAseq.copy()

# Make sure routeId is string
df["routeId"] = df["routeId"].astype(str)

# If generationRun is missing, infer it from searchDepthUsed
if "generationRun" not in df.columns and "searchDepthUsed" in df.columns:
    df["generationRun"] = df["searchDepthUsed"]

# If reconstructedPathwayString is missing but old column exists
if "reconstructedPathwayString" not in df.columns and "multiStepReactionSMILES" in df.columns:
    df = df.rename(columns={"multiStepReactionSMILES": "reconstructedPathwayString"})


# One row per unique reconstructed pathway route
aggDict = {
    "dirName": ("dirName", "first"),
    "jobName": ("jobName", "first"),
    "numSteps": ("numSteps", "first"),
    "searchDepthUsed": ("searchDepthUsed", "first"),
    "starterSMILES": ("starterSMILES", "first"),
    "targetSMILES": ("targetSMILES", "first"),
    "reconstructedPathwayString": ("reconstructedPathwayString", "first"),
}

if "sourceFolderNum" in df.columns:
    aggDict["sourceFolderNum"] = ("sourceFolderNum", "first")

if "generationRun" in df.columns:
    aggDict["generationRun"] = ("generationRun", "first")

completeRouteLevelDF = (
    df
    .sort_values(["dirName", "routeId", "step"])
    .groupby("routeId", as_index=False)
    .agg(**aggDict)
)


# Count 1-step, 2-step, and 3-step pathways per directory
indexCols = ["dirName", "jobName", "starterSMILES", "targetSMILES"]

if "generationRun" in completeRouteLevelDF.columns:
    indexCols.insert(2, "generationRun")

starterTargetPathwayCountsDF_wOptimizedDNAseq = (
    completeRouteLevelDF
    .pivot_table(
        index=indexCols,
        columns="numSteps",
        values="routeId",
        aggfunc="nunique",
        fill_value=0,
    )
    .rename(columns={
        1: "numOneStepPathways",
        2: "numTwoStepPathways",
        3: "numThreeStepPathways",
    })
    .reset_index()
)

for c in ["numOneStepPathways", "numTwoStepPathways", "numThreeStepPathways"]:
    if c not in starterTargetPathwayCountsDF_wOptimizedDNAseq.columns:
        starterTargetPathwayCountsDF_wOptimizedDNAseq[c] = 0

starterTargetPathwayCountsDF_wOptimizedDNAseq["totalStarterToTargetPathways"] = (
    starterTargetPathwayCountsDF_wOptimizedDNAseq["numOneStepPathways"]
    + starterTargetPathwayCountsDF_wOptimizedDNAseq["numTwoStepPathways"]
    + starterTargetPathwayCountsDF_wOptimizedDNAseq["numThreeStepPathways"]
)

starterTargetPathwayCountsDF_wOptimizedDNAseq = (
    starterTargetPathwayCountsDF_wOptimizedDNAseq
    .sort_values("totalStarterToTargetPathways", ascending=False)
    .reset_index(drop=True)
)


print(f"Total reaction-step rows              : {len(df):,}")
print(f"Unique starter --> target pathways   : {completeRouteLevelDF['routeId'].nunique():,}")
print(f"Directories with pathways            : {starterTargetPathwayCountsDF_wOptimizedDNAseq['dirName'].nunique():,}")

print("\nPathways by step count:")
print(completeRouteLevelDF["numSteps"].value_counts().sort_index())

starterTargetPathwayCountsDF_wOptimizedDNAseq

### Plot reaction pathways

Renders each route as a stacked figure (starter → reaction steps → target) and saves a JPG and PDF per route to the DNA-design folder. Requires Pillow (imported in the setup cell).

In [ ]:
def splitPathway(pathwayStr):
    return [] if pd.isna(pathwayStr) else [x.strip() for x in str(pathwayStr).split("||") if x.strip()]


def splitStarters(starterStr):
    return [] if pd.isna(starterStr) else [x.strip() for x in str(starterStr).split(";") if x.strip()]


def safeName(rawName):
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(rawName))


def rdkitImageToPil(imageObj):
    if isinstance(imageObj, Image.Image):
        return imageObj.convert("RGB")
    if isinstance(imageObj, (bytes, bytearray)):
        return Image.open(BytesIO(imageObj)).convert("RGB")
    return Image.open(BytesIO(imageObj.data)).convert("RGB")


def makeTextBanner(textValue, bannerWidth, bannerHeight=36, bgColor="white", fgColor="black"):
    bannerImage = Image.new("RGB", (bannerWidth, bannerHeight), bgColor)
    drawer = ImageDraw.Draw(bannerImage)
    drawer.text((10, 10), textValue, fill=fgColor)
    return bannerImage


def makeMolGridPanel(smilesList, panelTitle, molsPerRow=3, subImageSize=(300, 220)):
    molList = [Chem.MolFromSmiles(s) for s in smilesList if s]
    molList = [m for m in molList if m is not None]
    if len(molList) == 0:
        return None

    gridImage = Draw.MolsToGridImage(molList, molsPerRow=molsPerRow, subImgSize=subImageSize)
    gridImage = rdkitImageToPil(gridImage)

    titleImage = makeTextBanner(panelTitle, gridImage.width, bannerHeight=34)
    return ImageOps.expand(titleImage, border=0), ImageOps.expand(gridImage, border=1, fill="black")


def makeReactionPanel(rxnSmiles, panelTitle, subImageSize=(460, 280)):
    reactionObj = AllChem.ReactionFromSmarts(str(rxnSmiles), useSmiles=True)
    if reactionObj is None:
        return None

    reactionImage = Draw.ReactionToImage(reactionObj, subImgSize=subImageSize)
    reactionImage = rdkitImageToPil(reactionImage)

    titleImage = makeTextBanner(panelTitle, reactionImage.width, bannerHeight=34)
    return ImageOps.expand(titleImage, border=0), ImageOps.expand(reactionImage, border=1, fill="black")


def stackPanelsVertically(panelList, gapSize=10, bgColor="white"):
    validPanels = [p for p in panelList if p is not None]
    if not validPanels:
        return None

    canvasWidth = max(p.width for p in validPanels)
    canvasHeight = sum(p.height for p in validPanels) + gapSize * (len(validPanels) - 1)

    canvasImage = Image.new("RGB", (canvasWidth, canvasHeight), bgColor)
    yOffset = 0
    for panelImage in validPanels:
        xOffset = (canvasWidth - panelImage.width) // 2
        canvasImage.paste(panelImage, (xOffset, yOffset))
        yOffset += panelImage.height + gapSize

    return canvasImage


for _, rowData in completeRouteLevelDF.iterrows():
    dirNameSafe = safeName(rowData["dirName"])

    panelBlocks = []

    headerText = (
        f"{rowData['dirName']} | Job: {rowData['jobName']} | Route: {rowData['routeId']} | "
        f"Gen: {rowData['generationRun']} | Steps: {rowData['numSteps']}"
    )
    headerPanel = makeTextBanner(headerText, bannerWidth=1600, bannerHeight=42)
    panelBlocks.append(headerPanel)

    starterSmilesList = splitStarters(rowData.get("starterSMILES", np.nan))
    starterPanel = makeMolGridPanel(
        starterSmilesList,
        "Starter molecule",
        molsPerRow=3,
        subImageSize=(300, 220),
    )
    if starterPanel:
        panelBlocks.extend(starterPanel)

    pathwaySteps = splitPathway(rowData.get("reconstructedPathwayString", np.nan))
    for stepIdx, reactionSmiles in enumerate(pathwaySteps, start=1):
        stepPanel = makeReactionPanel(
            reactionSmiles,
            f"Reaction step {stepIdx}",
            subImageSize=(460, 280),
        )
        if stepPanel:
            panelBlocks.extend(stepPanel)

    targetPanel = makeMolGridPanel(
        [rowData.get("targetSMILES", "")],
        "Target molecule",
        molsPerRow=1,
        subImageSize=(320, 240),
    )
    if targetPanel:
        panelBlocks.extend(targetPanel)

    finalImage = stackPanelsVertically(panelBlocks, gapSize=10, bgColor="white")
    if finalImage is None:
        continue

    jpgOutputPath = os.path.join(DNADesignResultsDir, f"{dirNameSafe}.jpg")
    pdfOutputPath = os.path.join(DNADesignResultsDir, f"{dirNameSafe}.pdf")

    finalImage.save(jpgOutputPath, "JPEG", quality=95)
    finalImage.save(pdfOutputPath, "PDF", resolution=300.0)

    display(finalImage)

print("Saved route images to:", DNADesignResultsDir)